# Matched thinking OFF/ON follow-up, six runs - disabled

Frozen S derivatives: both reserve 4096 total completion tokens. OFF is a new
baseline, not the submitted agent. ON requests thinking enabled with budget 1024.
Only sampling.yaml differs; prompts/tools/analyzer are identical. Two selected Rich
development tasks, six runs; heldout excluded. rich_3278 is REPEATED from the
four-run comparison with reversed order, so it is a reproducibility check and NOT an
independent new task; rich_3535 and rich_3942 are new to this pair. No leaderboard extrapolation.
32768 context minus 4096 completion permits at most 28672 prompt tokens; neither
compaction nor a thinking budget guarantees the next request fits or reserves
final-tool-call space. Compaction 20480 is equal across arms and differs from the
public harness default None; private grader configuration is unknown.
Forwarding is offline-verified; host enforcement is unverified. Inspect recorded
thought/reasoning evidence if present; absence is unknown, not proof of disabled
thinking. Total completion tokens do not prove a thinking budget is enforced.
All ungraded, truncated, failed and missing runs stay visible. Session cap is an
admission limit, not a hard kill. No follow-on launch or competition submission.


In [ ]:
# ============================ CONFIG ============================
DISPATCH_CONFIRM = True     # guards model startup/dispatch; a Kaggle GPU session still consumes quota during setup
SANDBOX_ROOT_STR = "/tmp"    # where swegemma creates its sandbox directories
CLEANUP_GRACE_S  = 45        # bounded wait for THIS run's sandboxes to be torn down
SESSION_CAP_MIN  = 210       # admission deadline; not a hard OS-process/session kill
RUN_RESERVE_MIN = 25         # refuse a new run when less than this planning allowance remains

# Identical for A and B. Not tuned per variant.
BUDGET = dict(max_time_minutes=10.0, max_tool_calls=100, max_turns=60, timeout_seconds=300)
# ================================================================
import json, os, sys, time
from pathlib import Path
T0 = time.perf_counter()
def elapsed_min():
    return (time.perf_counter() - T0) / 60.0
print("thinking_v3 follow-up:", BUDGET, "| session cap", SESSION_CAP_MIN, "min")
if not DISPATCH_CONFIRM:
    print("DISPATCH_CONFIRM is False: setup and preconditions will run, the six runs will NOT.")


In [ ]:
# Wheelhouse install. Exit status is checked, not piped away.
import glob, importlib, os, subprocess, sys
from pathlib import Path
os.environ.update({
    "LITELLM_LOCAL_MODEL_COST_MAP": "True", "TRANSFORMERS_NO_TF": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200", "VLLM_NO_USAGE_STATS": "1",
    "OTEL_SDK_DISABLED": "true", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
            "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path("/tmp/wheelhouse"); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
print("installing", len(wheels), "wheels...")
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                       "--force-reinstall", *wheels], capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stdout[-2000:]); print(proc.stderr[-2000:])
    raise SystemExit(f"wheelhouse install failed rc={proc.returncode}")
importlib.invalidate_caches()
import importlib.metadata as md
VERSIONS = {}
for p in ("swegemma", "adk-submission", "adk-eval-core", "google-adk", "vllm", "transformers", "litellm"):
    try:
        VERSIONS[p] = md.version(p)
    except Exception as e:
        VERSIONS[p] = "MISSING: " + repr(e)[:60]
print(json.dumps(VERSIONS, indent=2))


In [ ]:
# Hardware / model identity.
import hashlib, json, torch
from pathlib import Path
DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2")
WORKING_DIR = Path("/kaggle/working"); WORKING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = WORKING_DIR / "pilot"; RESULTS.mkdir(parents=True, exist_ok=True)

N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
print("GPUs:", N_GPU, GPU_NAMES)
assert MODEL_PATH.exists(), "attach model gemma-4-31b-it-qat-w4a16-ct/2"
assert DATA_DIR.exists(), "attach the competition data"
_cfg_bytes = (MODEL_PATH / "config.json").read_bytes()
MODEL_CONFIG_SHA = hashlib.sha256(_cfg_bytes).hexdigest()
_cfg = json.loads(_cfg_bytes)
QUANT = _cfg.get("quantization_config") or _cfg.get("text_config", {}).get("quantization_config")
print("model revision dir:", MODEL_PATH.name, "| config sha256:", MODEL_CONFIG_SHA[:16])
print("architectures:", _cfg.get("architectures"))
print("quantization:", json.dumps(QUANT)[:200])
if N_GPU != 4:
    print(f"WARNING: {N_GPU} GPU(s); the grader uses 4x L4. Timings are not comparable.")


In [ ]:
# Shared helpers. Defined BEFORE any cell that uses them (the previous notebook referenced
# run_sync / TASK_IDS from the repair cell, which were only defined later -> NameError at runtime).
import asyncio, concurrent.futures, re
from swegemma.models import load_tasks

TASKS_PATH = DATA_DIR / "tasks.jsonl"
all_tasks = load_tasks(TASKS_PATH)
ALL_BY_ID = {t.instance_id: t for t in all_tasks}
PKG = {"fastapi/fastapi": "fastapi", "Textualize/rich": "rich",
       "psf/requests": "requests", "encode/httpx": "httpx"}

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def snapshot_for(t):
    return DATA_DIR / "snapshots" / (t.instance_id + ".tgz")

def target_tests(t):
    return re.findall(r"^\+\+\+ b/(\S+)", t.test_patch or "", re.M)

def pkg_of(t):
    return PKG.get(t.repo, t.repo.split("/")[-1])

def under(path, ws):
    from pathlib import PurePosixPath
    import posixpath
    if not path or not ws or not path.startswith('/') or not ws.startswith('/'):
        return False
    actual, root = PurePosixPath(posixpath.normpath(path)), PurePosixPath(posixpath.normpath(ws))
    return actual != root and actual.is_relative_to(root)

print("loaded", len(all_tasks), "tasks")


import time
from pathlib import Path

def sandbox_set():
    '''Sandbox directories currently on disk. Used to attribute cleanup to a specific run.'''
    return {str(p) for p in Path(SANDBOX_ROOT_STR).glob('swegemma_sandbox_*')}


def confirm_cleanup(before, grace_s=None):
    '''Sandboxes THIS run created that are still present after a bounded grace period.

    Only directories absent before the run are counted, so an unrelated leftover cannot mask or
    manufacture a failure. Nothing is ever deleted here: a leak is reported, not tidied away, and an
    unrelated sandbox is never touched. Returns (ok, owned_leftovers).
    '''
    grace_s = CLEANUP_GRACE_S if grace_s is None else grace_s
    deadline = time.time() + grace_s
    owned = sorted(sandbox_set() - before)
    while owned and time.time() < deadline:
        time.sleep(3)
        owned = sorted(sandbox_set() - before)
    return (not owned), owned


In [ ]:
# Task manifest + the saved control evidence these tasks were selected on.
TASK_IDS = ["rich_3278", "rich_3535", "rich_3942"]
CONTROL_EVIDENCE = {'rich_3278': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_fbc86660-ea1_mbw02i_a/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_fbc86660-ea1_mbw02i_a/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_ansi::test_decode': 'passed', 'test_ansi::test_decode_example': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]': 'passed', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]': 'passed', 'test_ansi::test_strip_private_escape_sequences[0]': 'failed', 'test_ansi::test_strip_private_escape_sequences[1]': 'failed', 'test_ansi::test_strip_private_escape_sequences[2]': 'failed', 'test_ansi::test_strip_private_escape_sequences[3]': 'failed', 'test_ansi::test_strip_private_escape_sequences[4]': 'failed', 'test_ansi::test_strip_private_escape_sequences[5]': 'failed', 'test_ansi::test_strip_private_escape_sequences[6]': 'failed', 'test_ansi::test_strip_private_escape_sequences[7]': 'failed', 'test_ansi::test_strip_private_escape_sequences[8]': 'failed', 'test_ansi::test_strip_private_escape_sequences[9]': 'failed', 'test_ansi::test_strip_private_escape_sequences[:]': 'failed', 'test_ansi::test_strip_private_escape_sequences[;]': 'failed', 'test_ansi::test_strip_private_escape_sequences[<]': 'failed', 'test_ansi::test_strip_private_escape_sequences[=]': 'failed', 'test_ansi::test_strip_private_escape_sequences[>]': 'failed', 'test_ansi::test_strip_private_escape_sequences[?]': 'failed'}, 'n_cases': 23, 'failed_nodes': ['test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'passed_nodes': ['test_ansi::test_decode', 'test_ansi::test_decode_example', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]'], 'failed': ['test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_2c91183e-840_38s9ivy0/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_2c91183e-840_38s9ivy0/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_ansi::test_decode': 'passed', 'test_ansi::test_decode_example': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]': 'passed', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]': 'passed', 'test_ansi::test_strip_private_escape_sequences[0]': 'passed', 'test_ansi::test_strip_private_escape_sequences[1]': 'passed', 'test_ansi::test_strip_private_escape_sequences[2]': 'passed', 'test_ansi::test_strip_private_escape_sequences[3]': 'passed', 'test_ansi::test_strip_private_escape_sequences[4]': 'passed', 'test_ansi::test_strip_private_escape_sequences[5]': 'passed', 'test_ansi::test_strip_private_escape_sequences[6]': 'passed', 'test_ansi::test_strip_private_escape_sequences[7]': 'passed', 'test_ansi::test_strip_private_escape_sequences[8]': 'passed', 'test_ansi::test_strip_private_escape_sequences[9]': 'passed', 'test_ansi::test_strip_private_escape_sequences[:]': 'passed', 'test_ansi::test_strip_private_escape_sequences[;]': 'passed', 'test_ansi::test_strip_private_escape_sequences[<]': 'passed', 'test_ansi::test_strip_private_escape_sequences[=]': 'passed', 'test_ansi::test_strip_private_escape_sequences[>]': 'passed', 'test_ansi::test_strip_private_escape_sequences[?]': 'passed'}, 'n_cases': 23, 'failed_nodes': [], 'passed_nodes': ['test_ansi::test_decode', 'test_ansi::test_decode_example', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]', 'test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'failed': [], 'errored': []}}, 'rich_3535': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_b9bb1fae-ea7_j4kmphm5/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_b9bb1fae-ea7_j4kmphm5/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_cells::test_cell_len_long_string': 'passed', 'test_cells::test_cell_len_short_string': 'passed', 'test_cells::test_set_cell_size': 'passed', 'test_cells::test_set_cell_size_infinite': 'passed', 'test_cells::test_chop_cells': 'passed', 'test_cells::test_chop_cells_double_width_boundary': 'passed', 'test_cells::test_chop_cells_mixed_width': 'passed', 'test_cells::test_is_single_cell_widths': 'failed'}, 'n_cases': 8, 'failed_nodes': ['test_cells::test_is_single_cell_widths'], 'passed_nodes': ['test_cells::test_cell_len_long_string', 'test_cells::test_cell_len_short_string', 'test_cells::test_chop_cells', 'test_cells::test_chop_cells_double_width_boundary', 'test_cells::test_chop_cells_mixed_width', 'test_cells::test_set_cell_size', 'test_cells::test_set_cell_size_infinite'], 'failed': ['test_cells::test_is_single_cell_widths'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_613d24b0-112_yqshur97/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_613d24b0-112_yqshur97/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_cells::test_cell_len_long_string': 'passed', 'test_cells::test_cell_len_short_string': 'passed', 'test_cells::test_set_cell_size': 'passed', 'test_cells::test_set_cell_size_infinite': 'passed', 'test_cells::test_chop_cells': 'passed', 'test_cells::test_chop_cells_double_width_boundary': 'passed', 'test_cells::test_chop_cells_mixed_width': 'passed', 'test_cells::test_is_single_cell_widths': 'passed'}, 'n_cases': 8, 'failed_nodes': [], 'passed_nodes': ['test_cells::test_cell_len_long_string', 'test_cells::test_cell_len_short_string', 'test_cells::test_chop_cells', 'test_cells::test_chop_cells_double_width_boundary', 'test_cells::test_chop_cells_mixed_width', 'test_cells::test_is_single_cell_widths', 'test_cells::test_set_cell_size', 'test_cells::test_set_cell_size_infinite'], 'failed': [], 'errored': []}}, 'rich_3942': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_cb7391fe-ddf_sauv1peb/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_cb7391fe-ddf_sauv1peb/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_markdown::test_markdown_render': 'failed', 'test_markdown::test_inline_code': 'passed', 'test_markdown::test_markdown_table': 'failed', 'test_markdown::test_inline_styles_in_table': 'failed', 'test_markdown::test_inline_styles_with_justification': 'failed', 'test_markdown::test_partial_table': 'failed', 'test_markdown::test_table_with_empty_cells': 'passed', 'test_markdown_no_hyperlinks::test_markdown_render': 'failed'}, 'n_cases': 8, 'failed_nodes': ['test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown_no_hyperlinks::test_markdown_render'], 'passed_nodes': ['test_markdown::test_inline_code', 'test_markdown::test_table_with_empty_cells'], 'failed': ['test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown_no_hyperlinks::test_markdown_render'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_58f2d1f1-b18_zzqdtsd6/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_58f2d1f1-b18_zzqdtsd6/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_markdown::test_markdown_render': 'passed', 'test_markdown::test_inline_code': 'passed', 'test_markdown::test_markdown_table': 'passed', 'test_markdown::test_inline_styles_in_table': 'passed', 'test_markdown::test_inline_styles_with_justification': 'passed', 'test_markdown::test_partial_table': 'passed', 'test_markdown::test_table_with_empty_cells': 'passed', 'test_markdown_no_hyperlinks::test_markdown_render': 'passed'}, 'n_cases': 8, 'failed_nodes': [], 'passed_nodes': ['test_markdown::test_inline_code', 'test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown::test_table_with_empty_cells', 'test_markdown_no_hyperlinks::test_markdown_render'], 'failed': [], 'errored': []}}}
EXPECTED_TASK_HASHES = {"rich_3278": "8fb8127fdb3e069ae02db8a0e3d2e5306b96e7d42a1ab7c3185da2e269cd61bf", "rich_3535": "3c70ac67cc8e78d329459615dd6db711e2715b92f6da4632cc4905f1faf0ad18", "rich_3942": "b626009683a3a55af3ad72e8a4cc9cd629cb24444a2c404c073a87778b8148b8"}
import hashlib
def task_hash(t):
    return hashlib.sha256(json.dumps({k:getattr(t,k,None) for k in
        ('repo','base_commit','problem_statement','hints_text','patch','test_patch')},sort_keys=True).encode()).hexdigest()

missing = [t for t in TASK_IDS if t not in ALL_BY_ID]
assert not missing, f"unknown task ids: {missing}"
SELECTED = [ALL_BY_ID[t] for t in TASK_IDS]
for t in SELECTED:
    assert task_hash(t) == EXPECTED_TASK_HASHES.get(t.instance_id), f'{t.instance_id}: task content changed since local preparation; rerun controls'

print("PILOT TASKS and the evidence they were chosen on:")
for tid in TASK_IDS:
    ev = CONTROL_EVIDENCE.get(tid)
    print("\n ", tid, ALL_BY_ID[tid].repo)
    if not ev:
        raise AssertionError(f"{tid}: missing saved control evidence")
    for phase in ("negative", "positive"):
        e = ev.get(phase, {})
        print(f"    {phase:9s} exit={e.get('pytest_exit')} cases={e.get('n_cases')} "
              f"failed={len(e.get('failed', []))} errored={len(e.get('errored', []))}")
        print(f"              agent  import: {e.get('agent_pkg_file','')}")
        print(f"              pytest import: {e.get('pytest_pkg_file','')}")
    neg, pos = ev.get("negative", {}), ev.get("positive", {})
    failed = set(neg.get('failed_nodes', []))
    passed = set(pos.get('passed_nodes', []))
    ok = (neg.get('pytest_exit') == 1 and pos.get('pytest_exit') == 0
          and bool(failed) and failed <= passed
          and not neg.get('errored') and not pos.get('errored')
          and all(under(e.get(key, ''), e.get('workspace_real', ''))
                  for e in (neg,pos) for key in ('pytest_pkg_file',)))
    print(f"    baseline-fails / gold-passes / grading-import-in-checkout: {ok}")
    print("    agent-side import provenance: UNAVAILABLE (no agent ran in the control)")
    print(f"    control patch_rc={neg.get('patch_rc')} test_patch_rc={neg.get('test_patch_rc')}")
    assert ok, f"{tid}: saved control evidence does not support using this task"
    if neg.get("failed"):
        print(f"    baseline failures: {neg['failed'][:4]}")


In [ ]:
# The agent prompt must contain no answer-key text beyond what the issue itself carries.
from swegemma.harness.agent_runner import build_agent_prompt
from dataclasses import replace

class _B:
    time_minutes, tool_calls, turns, cost_usd = (BUDGET["max_time_minutes"], BUDGET["max_tool_calls"],
                                                 BUDGET["max_turns"], None)
class _H:
    command_timeout_seconds, max_stdout_chars, max_file_lines, max_file_chars = (
        BUDGET["timeout_seconds"], 5000, 150, 10000)
class _C:
    budget, harness = _B(), _H()
    graph_dir, embeddings_dir = str(DATA_DIR / "graphs"), str(DATA_DIR / "embeddings")
    enable_sandbox_testing = True

def added_lines(diff):
    return [l[1:].strip() for l in (diff or "").splitlines()
            if l.startswith("+") and not l.startswith("+++") and len(l.strip()) > 40]

leaks, benign = [], 0
for t in SELECTED:
    prompt = build_agent_prompt(task=t, config=_C(), workspace_tree="")
    assert prompt == build_agent_prompt(task=replace(t, patch="SENTINEL", test_patch="SENTINEL"),
                                        config=_C(), workspace_tree=""), "answer keys affect the prompt"
    issue = (t.problem_statement or "") + "\n" + (getattr(t, "hints_text", "") or "")
    for src in ("patch", "test_patch"):
        for l in added_lines(getattr(t, src, "")):
            if l in prompt:
                benign += 1 if l in issue else 0
                if l not in issue:
                    leaks.append((t.instance_id, src, l[:70]))
print("prompt leakage:", "CLEAN" if not leaks else leaks[:3], f"({benign} lines came from the issue text)")
assert not leaks
print("LIMITATION: this is a data-flow check. The subprocess backend runs run_command on the host, so")
print("answer-key files remain reachable; traces must be audited for contamination before trusting a run.")


In [ ]:
# Candidates A and B, embedded as frozen zips.
import base64, hashlib, io, shutil, zipfile
BUNDLES = {"OFF": {"b64": "UEsDBBQAAAAIAAAAIQCMhuWn/QAAAJsBAAAKAAAAYWdlbnQueWFtbE1QW27DMAz7zym0A7hFsWIfOcCuYai2kgrzI7PkZunpZycDOn8YoERRpBJGGkFWsi57KkNsfxhhphjRXM375WZYzTeqWa94+TBOB0/iCi/KOY3wyT8kkBNBoSULay4bsEglWFnvgPCgwhOTB8m1OIIF1d1PAyfRUt2h8sbJhepbs+S4qJxlE6V4in7QnIOMA4CBUlNz2Ywlf2BCbycOtCPyrC+0FlZ6wZnUiqJW2aHUW2zs3cpewJmS2r6rr+rP5TTx3Cn3sfPtTpFzP5PFhGF7UjltGMPfgHzxYqU2e4WfeORqCWloc1RQ+4WT9jWH9L/UR6GlxrgETvMh+wtQSwMEFAAAAAgAAAAhAJzTq+JsAAAAiwAAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWw9jFEKwyAMQP89RU4wytgG9TLB1VTFNpGawI5fO8b+Hg/eU9obHUHtIA/T7e5UGrYL5+eXq4fH5PbwQTFtpqhSiftl55frRHHgyHLhWjjhIryW5B3AX70tJtJfAVB42SwSahZLWcdqDVsndwJQSwMEFAAAAAgAAAAhABAbPpIjAwAAaAUAABMAAABwcm9tcHRzL2FuYWx5emVyLm1kVVTRbttGEHznVyyShziAxCRun2RDgOvIiJHUDWq1aJ7EE7kkDz7eMbdHyyr68Zk9ykn6IIk47c3szs7wS5jIRKaqDg3vjDfu+C/HakGGIptmGbw7kv5H3jzazqQQqcUn9YyCMYjFyZFMojeHEB9kNDWX9AWonh850hAa2x6ptY5lPo9cs31kMp6syMQLOvS4roC1cQ53eiNknNIfybGJnhv04xv01IZ6Em7o68SSbPAl/TZZ11DwAAjCIPKN9Z2Q9ZKAQKHVPtkknCrJcILycgDVwaYe4/k6cuLcJblQG4WWXMePtmGPmYri5UvahuCkWFIVJ7+rwzCgpMp6/CxWPpYVVZ1N1IGdlh6KVvNjzM86xNLTq/vt1Z/bxebu/fiKbm4/bfQ/J7kal50ss3JVSbHL/aTInBfmQ8ozqmRNmXtCCzstr+axku16QFiPTRnfsVAKOmtr4/Bdp3yz47TL+/eMO/sQpaI3Px3LtO+iGftqRXL0dR+DD5PQC92XvCBuFFyHvyCjBdROvj5piFYH7FnVb2MY8p4zWGYWrLfud2IH60zMbBUl8wA8A65hHxyMNzCdcdmVVF19vv3Ixw+YFCZ9vcgqoJB90iVdkE2iwz4onYUJhhF2s8JKdtuiNFNDCOByjJh0QZLCSFPuEJKryHNXuVUJU3ze/u8MjzXFu5I2TymaGguAN5JtLUf5MV129Sq3DXR8ien0aTSpx08YszJlcV7S/Ux0CtSQydvgXDjMedDcYWmHniHjnntEEP1Qg/xE1Txzfo9PJiZ+GrlOwP+lpBurFn6yMrsfmZETGa4oeln8WtL1yRMaWPjXGTvQ/pgtfQpNrr04RbpD9mT2lZ+GPWYv6S9ED5BDECi4D1Oid+ezytkjs35Xc+bOngvP374lvDQayXvslYud8OtCg3C/oktVbL26hMljWi8v2TdrOrucbbHG9gNaGIGovRSbv2/fb+6uN7iXFbH+h4bShwPs36slZH535XCMDq+rNfj+wS2ZOmiaEMy617ysi+3mfqt9/E/B+V12khFoSrAurv+4u3nm74FN/2HzjZ0GPGCdxTdQSwMEFAAAAAgAAAAhAA//CbpnBQAAJgoAABEAAABwcm9tcHRzL3N5c3RlbS5tZHVW227cNhB9368YOEXcGivZubgPa9dFWjho0CIpcmmapxVXolaEJZIhqb0U/fieGWrX2wJ9sSlez8ycc2a/uJFU0KQsqTE56wY3RoraGhfo933qnCVt18ZqHSjo6PqNsWtyVpOJccRfS6nTWPIumuTCnlSiy60LD9GrWpf0GUPamtTR3o2BknN9pNEm08vB6ZZIrdnpZs5zlmrV91TFcTWYtPQq1V1Vzma/mKbBYtIxRQojEK+VsTHli+tO2bWODEhRC6gdpnT94MZU0sdO78mrGIG835Np5e3aNZpWulMbnFPxBI+KD3GOq+p+bDheXtE7VSeyatBY8ipgkHTAGBNRrXk27T3fZBuKSSUkkl8ApIQ9iNjZWNLFxUcEgHB73ho4dVEnwGgdPtZB8YNzim6KCKMx1BNYRj8nqzeohuRhTpXf86g01lSEolW1s63M+H1VXlzMZk+eSBHa3m1nzxjAJ9sAeAJOrNMHteekcUlxPG0dlz9pWwPftkM1H9Oid17XKUcoS53yXltOekxaNSV9cMO0OQfnx74vgv46cszIRR2Ml0QsKD4YL3cnPfheJR7sUjl7zhB/c+CA+UszwGqNBK7BMCosFa+pKOjs1jSc0dYgEW1wwyPIuzPecH6B8M/pb+oAi4oXV9VErcDfvHlgWnFtQ+aNULTi5SXXBoR7wUDeax9cM9ac+1rgfA4maUkX+KQDR1OsVNQoukQn2WF6GhDvnRVtyBW8k5murRvX3UIuZG60yvSZfFuU4jHbOV0rJlZMzh/vYdATxwITaYeJcvaS0d43Jtcrk4bhfoqaKo35HFaOU1HsHA6P1qA0VLm+WcYUcFEFonmDYITu0Moxu3w8B9S7tUF5DgT1qAE/UM6uGcMfOph2L4XzYiAvqBjI75e1GzzuoFu+6e5YkDE7CIJZQ1CN8FoeO1as4D0i8tNUlrPv+bkP4hL83M8s98yWxrQtaMBiLVgzVUnvca0Tp9vnJ+Ar4kmwpBFO0UwM4OMH42Am3fK+u0o85P+cCQq7txsTnGWdzwp617Y9PPMGNYQ+QNXaTIJgpeAODa28FRlXHjKYZvHKqw3YoFa9XhCQzIX3c6TDwhsjG6TaPsB+cmI5/rcOjHs8FOAdKWguPeMYk0cYIF2NfxDs9fzq6ooLF1BdmIAQ6YT1yHAaAxSNvYNDlp5dXxGHEjmDkE7r6pG5nlVzQ0DPhOCekN9KXWB6U8XKQ3IKrlEdOFWT6a0d+81lGhCYPL9lReX3L0+pWitrEZ0sMxfCkdCK+Ktx9WJGcDyAvZMLL4UgkD7d3p7fv3t9XlFZllRhWB0S3mv4PSvgBJQAOjatBatypcFXoPUsMtdm/knFOaafYbImDMIfLisz6tAojuyZPD2FMS8q9vqxTwtAbk7eo6dP6SiVms7M4Fmct/5hfYcMQ5bpW/kol5KZ5fK7s4qGkUnMi7hYyDyyr582XwD9VcM3c4s4h/XsmN3oJIv/iFMUkYVJxVcqdoeEcvtMu0TP754+uyFdd47O7v988/GHb348u4Fo0caLl1f/2luJE/FL1+h7CmSyburaW1QN74P/YB/IzOWNY41WEzMXlLQFf7CBxukomzZiKizexKZ96Gs/jQ1MY/YFVehyVXsDZeK4HYcVkoHCCZ/RmViIXJ/h8JvFBU6XWrkN0iFkd23eAJ9GP/ORXgqo6wz0OPtcZqG+N0KL/LSEwo7M3gkOc8Gn1r3ak2iiz+7eqb5lp/cOtZvL5syuFYdcjyHARKjbe4fd0WSNTgngnvIqf7BJMzkNnBhNAKeVtGbQIPsTIzF2w8xcq+yYM6HDZKUqOisopRVI9utH/fIPNbG7WM7+AVBLAwQUAAAACAAAACEAvcTrPiYBAADYAQAAHQAAAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sVVAxcsMwDNv9CnaPnMs11yF9QPd+wEdLjKw7iXJFyonz+spOO2TQAAggQTAmuoDNjgZkjOuDSpcaihfwlBKas3k/jSao+UE1tzOePozVzpHYEmYNmS/wTehM5rjuc4BxCR41F7i2N2FxRrOJ2WIMD4IgUkl6+ApLAwo6/XGgdFdAdnCbUGHNFTCWNnttgyq7z01dSGth2V2FIi3ICtcQSXanrGnMUQ5AS3DElg6AINV7EiXXhPfDrmtW3AhtvPRdYNFS7fOct8A21nZI3x/nktOscvzvpk+u09w2XDoAA6XyYHOrid0Tt7TDlmZHnnTYm2UKfhpzkZ0WwmKnQUIKEcuueJVLHX3Beeo8MZWWs9GsxNs3X4N/jfjk5CiY5hjY9yum2P0CUEsBAhQAFAAAAAgAAAAhAIyG5af9AAAAmwEAAAoAAAAAAAAAAAAAAKQBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAACEAnNOr4mwAAACLAAAAFQAAAAAAAAAAAAAApAElAQAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAAAhABAbPpIjAwAAaAUAABMAAAAAAAAAAAAAAKQBxAEAAHByb21wdHMvYW5hbHl6ZXIubWRQSwECFAAUAAAACAAAACEAD/8JumcFAAAmCgAAEQAAAAAAAAAAAAAApAEYBQAAcHJvbXB0cy9zeXN0ZW0ubWRQSwECFAAUAAAACAAAACEAvcTrPiYBAADYAQAAHQAAAAAAAAAAAAAApAGuCgAAc3ViX2FnZW50cy9jb2RlX2FuYWx5emVyLnlhbWxQSwUGAAAAAAUABQBGAQAADwwAAAAA", "sha256": "640fadab5b5b638a7e8a31abb26fcecb6b2e139929d77cdb6691f2d832ca2ec1"}, "ON": {"b64": "UEsDBBQAAAAIAAAAIQCMhuWn/QAAAJsBAAAKAAAAYWdlbnQueWFtbE1QW27DMAz7zym0A7hFsWIfOcCuYai2kgrzI7PkZunpZycDOn8YoERRpBJGGkFWsi57KkNsfxhhphjRXM375WZYzTeqWa94+TBOB0/iCi/KOY3wyT8kkBNBoSULay4bsEglWFnvgPCgwhOTB8m1OIIF1d1PAyfRUt2h8sbJhepbs+S4qJxlE6V4in7QnIOMA4CBUlNz2Ywlf2BCbycOtCPyrC+0FlZ6wZnUiqJW2aHUW2zs3cpewJmS2r6rr+rP5TTx3Cn3sfPtTpFzP5PFhGF7UjltGMPfgHzxYqU2e4WfeORqCWloc1RQ+4WT9jWH9L/UR6GlxrgETvMh+wtQSwMEFAAAAAgAAAAhAPIjp4lvAAAAigAAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWw9zEEKwzAMRNG9T6ETFDekhfgyIo1V27iRjCNBjl+ni+4+D2aU9kZ9VesUwN8mp9KwXbk8fl0DzN7t64li2kxRpRIfly5PdxDFkWOWC9fCCTfhd0nBAfzpZTGRBrj7aR5eePtYJNQslrKOK+1G7gtQSwMEFAAAAAgAAAAhABAbPpIjAwAAaAUAABMAAABwcm9tcHRzL2FuYWx5emVyLm1kVVTRbttGEHznVyyShziAxCRun2RDgOvIiJHUDWq1aJ7EE7kkDz7eMbdHyyr68Zk9ykn6IIk47c3szs7wS5jIRKaqDg3vjDfu+C/HakGGIptmGbw7kv5H3jzazqQQqcUn9YyCMYjFyZFMojeHEB9kNDWX9AWonh850hAa2x6ptY5lPo9cs31kMp6syMQLOvS4roC1cQ53eiNknNIfybGJnhv04xv01IZ6Em7o68SSbPAl/TZZ11DwAAjCIPKN9Z2Q9ZKAQKHVPtkknCrJcILycgDVwaYe4/k6cuLcJblQG4WWXMePtmGPmYri5UvahuCkWFIVJ7+rwzCgpMp6/CxWPpYVVZ1N1IGdlh6KVvNjzM86xNLTq/vt1Z/bxebu/fiKbm4/bfQ/J7kal50ss3JVSbHL/aTInBfmQ8ozqmRNmXtCCzstr+axku16QFiPTRnfsVAKOmtr4/Bdp3yz47TL+/eMO/sQpaI3Px3LtO+iGftqRXL0dR+DD5PQC92XvCBuFFyHvyCjBdROvj5piFYH7FnVb2MY8p4zWGYWrLfud2IH60zMbBUl8wA8A65hHxyMNzCdcdmVVF19vv3Ixw+YFCZ9vcgqoJB90iVdkE2iwz4onYUJhhF2s8JKdtuiNFNDCOByjJh0QZLCSFPuEJKryHNXuVUJU3ze/u8MjzXFu5I2TymaGguAN5JtLUf5MV129Sq3DXR8ien0aTSpx08YszJlcV7S/Ux0CtSQydvgXDjMedDcYWmHniHjnntEEP1Qg/xE1Txzfo9PJiZ+GrlOwP+lpBurFn6yMrsfmZETGa4oeln8WtL1yRMaWPjXGTvQ/pgtfQpNrr04RbpD9mT2lZ+GPWYv6S9ED5BDECi4D1Oid+ezytkjs35Xc+bOngvP374lvDQayXvslYud8OtCg3C/oktVbL26hMljWi8v2TdrOrucbbHG9gNaGIGovRSbv2/fb+6uN7iXFbH+h4bShwPs36slZH535XCMDq+rNfj+wS2ZOmiaEMy617ysi+3mfqt9/E/B+V12khFoSrAurv+4u3nm74FN/2HzjZ0GPGCdxTdQSwMEFAAAAAgAAAAhAA//CbpnBQAAJgoAABEAAABwcm9tcHRzL3N5c3RlbS5tZHVW227cNhB9368YOEXcGivZubgPa9dFWjho0CIpcmmapxVXolaEJZIhqb0U/fieGWrX2wJ9sSlez8ycc2a/uJFU0KQsqTE56wY3RoraGhfo933qnCVt18ZqHSjo6PqNsWtyVpOJccRfS6nTWPIumuTCnlSiy60LD9GrWpf0GUPamtTR3o2BknN9pNEm08vB6ZZIrdnpZs5zlmrV91TFcTWYtPQq1V1Vzma/mKbBYtIxRQojEK+VsTHli+tO2bWODEhRC6gdpnT94MZU0sdO78mrGIG835Np5e3aNZpWulMbnFPxBI+KD3GOq+p+bDheXtE7VSeyatBY8ipgkHTAGBNRrXk27T3fZBuKSSUkkl8ApIQ9iNjZWNLFxUcEgHB73ho4dVEnwGgdPtZB8YNzim6KCKMx1BNYRj8nqzeohuRhTpXf86g01lSEolW1s63M+H1VXlzMZk+eSBHa3m1nzxjAJ9sAeAJOrNMHteekcUlxPG0dlz9pWwPftkM1H9Oid17XKUcoS53yXltOekxaNSV9cMO0OQfnx74vgv46cszIRR2Ml0QsKD4YL3cnPfheJR7sUjl7zhB/c+CA+UszwGqNBK7BMCosFa+pKOjs1jSc0dYgEW1wwyPIuzPecH6B8M/pb+oAi4oXV9VErcDfvHlgWnFtQ+aNULTi5SXXBoR7wUDeax9cM9ac+1rgfA4maUkX+KQDR1OsVNQoukQn2WF6GhDvnRVtyBW8k5murRvX3UIuZG60yvSZfFuU4jHbOV0rJlZMzh/vYdATxwITaYeJcvaS0d43Jtcrk4bhfoqaKo35HFaOU1HsHA6P1qA0VLm+WcYUcFEFonmDYITu0Moxu3w8B9S7tUF5DgT1qAE/UM6uGcMfOph2L4XzYiAvqBjI75e1GzzuoFu+6e5YkDE7CIJZQ1CN8FoeO1as4D0i8tNUlrPv+bkP4hL83M8s98yWxrQtaMBiLVgzVUnvca0Tp9vnJ+Ar4kmwpBFO0UwM4OMH42Am3fK+u0o85P+cCQq7txsTnGWdzwp617Y9PPMGNYQ+QNXaTIJgpeAODa28FRlXHjKYZvHKqw3YoFa9XhCQzIX3c6TDwhsjG6TaPsB+cmI5/rcOjHs8FOAdKWguPeMYk0cYIF2NfxDs9fzq6ooLF1BdmIAQ6YT1yHAaAxSNvYNDlp5dXxGHEjmDkE7r6pG5nlVzQ0DPhOCekN9KXWB6U8XKQ3IKrlEdOFWT6a0d+81lGhCYPL9lReX3L0+pWitrEZ0sMxfCkdCK+Ktx9WJGcDyAvZMLL4UgkD7d3p7fv3t9XlFZllRhWB0S3mv4PSvgBJQAOjatBatypcFXoPUsMtdm/knFOaafYbImDMIfLisz6tAojuyZPD2FMS8q9vqxTwtAbk7eo6dP6SiVms7M4Fmct/5hfYcMQ5bpW/kol5KZ5fK7s4qGkUnMi7hYyDyyr582XwD9VcM3c4s4h/XsmN3oJIv/iFMUkYVJxVcqdoeEcvtMu0TP754+uyFdd47O7v988/GHb348u4Fo0caLl1f/2luJE/FL1+h7CmSyburaW1QN74P/YB/IzOWNY41WEzMXlLQFf7CBxukomzZiKizexKZ96Gs/jQ1MY/YFVehyVXsDZeK4HYcVkoHCCZ/RmViIXJ/h8JvFBU6XWrkN0iFkd23eAJ9GP/ORXgqo6wz0OPtcZqG+N0KL/LSEwo7M3gkOc8Gn1r3ak2iiz+7eqb5lp/cOtZvL5syuFYdcjyHARKjbe4fd0WSNTgngnvIqf7BJMzkNnBhNAKeVtGbQIPsTIzF2w8xcq+yYM6HDZKUqOisopRVI9utH/fIPNbG7WM7+AVBLAwQUAAAACAAAACEAvcTrPiYBAADYAQAAHQAAAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sVVAxcsMwDNv9CnaPnMs11yF9QPd+wEdLjKw7iXJFyonz+spOO2TQAAggQTAmuoDNjgZkjOuDSpcaihfwlBKas3k/jSao+UE1tzOePozVzpHYEmYNmS/wTehM5rjuc4BxCR41F7i2N2FxRrOJ2WIMD4IgUkl6+ApLAwo6/XGgdFdAdnCbUGHNFTCWNnttgyq7z01dSGth2V2FIi3ICtcQSXanrGnMUQ5AS3DElg6AINV7EiXXhPfDrmtW3AhtvPRdYNFS7fOct8A21nZI3x/nktOscvzvpk+u09w2XDoAA6XyYHOrid0Tt7TDlmZHnnTYm2UKfhpzkZ0WwmKnQUIKEcuueJVLHX3Beeo8MZWWs9GsxNs3X4N/jfjk5CiY5hjY9yum2P0CUEsBAhQAFAAAAAgAAAAhAIyG5af9AAAAmwEAAAoAAAAAAAAAAAAAAKQBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAACEA8iOniW8AAACKAAAAFQAAAAAAAAAAAAAApAElAQAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAAAhABAbPpIjAwAAaAUAABMAAAAAAAAAAAAAAKQBxwEAAHByb21wdHMvYW5hbHl6ZXIubWRQSwECFAAUAAAACAAAACEAD/8JumcFAAAmCgAAEQAAAAAAAAAAAAAApAEbBQAAcHJvbXB0cy9zeXN0ZW0ubWRQSwECFAAUAAAACAAAACEAvcTrPiYBAADYAQAAHQAAAAAAAAAAAAAApAGxCgAAc3ViX2FnZW50cy9jb2RlX2FuYWx5emVyLnlhbWxQSwUGAAAAAAUABQBGAQAAEgwAAAAA", "sha256": "527acc5403d21a149ecea9d39d573d3d0f45dc65935d115952b772045339ea33"}}
CAND_ROOT = WORKING_DIR / "candidates"
assert CAND_ROOT.resolve().parent == WORKING_DIR.resolve(), 'Candidate cleanup must stay under working directory'
shutil.rmtree(CAND_ROOT, ignore_errors=True); CAND_ROOT.mkdir(parents=True)

def materialise(name, b64, want):
    payload = base64.b64decode(b64, validate=True)
    got = hashlib.sha256(payload).hexdigest()
    assert got == want, f"{name}: {got} != {want}"
    d = CAND_ROOT / name
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        assert z.testzip() is None and "agent.yaml" in z.namelist()
        z.extractall(d)
    return d

CAND_DIRS = {n: materialise(n, b["b64"], b["sha256"]) for n, b in BUNDLES.items()}
from adk_submission import validate_directory
from swegemma.config import build_submission_limits
from swegemma.models.discovery import validate_single_declared_model
LIMITS, GEN_CONSTRAINTS = build_submission_limits()
for n, d in CAND_DIRS.items():
    validate_directory(d, LIMITS)
    print(f"{n}: sha256={BUNDLES[n]['sha256'][:16]} model={validate_single_declared_model(d)}")
EXPECTED_FILES = {'OFF': {'agent.yaml': '8e890bec4c5e146402c6ef9ee22eb40388207ad8416cfd3405e040aa038ffb83', 'configs/sampling.yaml': 'a31b582a21cb5dbf93c9878dc8b42afde3d05140c715e78068353ab2ea5dba51', 'prompts/analyzer.md': '5011d37145d981e616d58fb56c5191ccbf336574fa01b3f65c2feab7d697b3a3', 'prompts/system.md': '39b2541c2220cc7655c576bb4ae7ac7e03200a72f2f1e518313068fc93be3b91', 'sub_agents/code_analyzer.yaml': '396b75ffd1315ef93f3d1806b814a49758b9f8dd8b8caef6a092bd6cb2c2d9bb'}, 'ON': {'agent.yaml': '8e890bec4c5e146402c6ef9ee22eb40388207ad8416cfd3405e040aa038ffb83', 'configs/sampling.yaml': '0b754a9811c6a40198102c45f7cee813f43e3fd97edfef2b6da5deeffb6fa3a8', 'prompts/analyzer.md': '5011d37145d981e616d58fb56c5191ccbf336574fa01b3f65c2feab7d697b3a3', 'prompts/system.md': '39b2541c2220cc7655c576bb4ae7ac7e03200a72f2f1e518313068fc93be3b91', 'sub_agents/code_analyzer.yaml': '396b75ffd1315ef93f3d1806b814a49758b9f8dd8b8caef6a092bd6cb2c2d9bb'}}
for key, folder in CAND_DIRS.items():
    got = {p.relative_to(folder).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
           for p in folder.rglob('*') if p.is_file()}
    assert got == EXPECTED_FILES[key], 'Candidate file drift: ' + key

EXPECTED_COMPILER_FILES = {'compiler.py': '39cc4884699faf147ca1b326664e888c1861f750fe36525b840b5f812d593eed', 'context.py': '3c77519415b1dd2cc97b42bcbe29d32dd85f8188ca7eba5b5fb8636bce92c908', 'discovery.py': 'ca19fc75fb3baa42e85438b41248a06c35987cbf26ef652159b302752d0f191c', 'errors.py': '7e121e7c6581288788500cd1a8deceb003a74ec150d5477c37f794971a5f4818', 'limits.py': 'afde5056f2df1fd77b53e0495712d03f60b3e246d4393617dd800d970f7482e2', 'paths.py': '7c4b7050141bdb05a9e6849be11f6440f2a18082a364623226c4343649b2f59f', 'registry.py': 'c17f9770c2f3663309dd61807d2ae4e7d75bbd06b7c674847f655d218e82cf7a', 'schema.py': '6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d', 'server.py': 'd60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb', 'yaml_loader.py': '01d644ef7f5acc216506245fd7e115e7096641e62e825c6873c83e148b23c69c', '__init__.py': 'ebe048f521fd5bcf46dad132f3e3dea2506df79bf471d560a70ce5b343ca387a', 'builders/llm.py': '97c27324ce60ac6dc466941bcc19f5482f7693af9fc02fb760a40526c84c10b5', 'builders/sub_agents.py': 'ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a', 'builders/workflows.py': 'd8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980', 'builders/__init__.py': 'ec6273c54cfb08f22777146c1636be4f13d4d00a67a17d9ae7a19e3a8f18f6c1', 'resolvers/callbacks.py': '332ce0732aa3eb1b555630c57683f51e59d5caee3f519371d376210e9ae33e29', 'resolvers/generation.py': 'ba8bf237261220d46843c8d71118faa387fc07f876f072756dff44d641dd1c14', 'resolvers/models.py': '39070f6ae52ad8880750b51af373da8762850d3a8ecc65c6f11744afe4cd5066', 'resolvers/tools.py': 'ae44a5ca815a05edfd763b92283d0444c77755afbd827fc35bcb87eb2633259e', 'resolvers/__init__.py': 'b88fe281c2b927a046e6485c7e0966b1808a5c8d0a0849cd85824ffb04ae53be'}
def verify_runtime_compiler(root=None, version=None):
    import importlib.metadata as metadata
    if root is None:
        import adk_submission
        root = Path(adk_submission.__file__).parent
    if version is None:
        version = metadata.version('adk-submission')
    found = {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in root.rglob('*.py')}
    record = {'version': version, 'python_files': found,
              'matches': version == '0.2.12' and found == EXPECTED_COMPILER_FILES}
    (RESULTS / 'compiler_identity.json').write_text(json.dumps(record, indent=2))
    assert record['matches'], 'Runtime compiler drift: refuse model startup'
verify_runtime_compiler()


In [ ]:
# ENVIRONMENT REPAIR (evaluation support only - never part of a submission artifact).
#
# swegemma/harness/container_setup.py returns immediately for the subprocess backend in
# install_editable_package (line 512) and install_test_dependencies (line 529). The sandbox venv
# carries _host_env.pth exposing the Kaggle host's dist-packages, which hold RELEASED
# fastapi/rich/requests/httpx, so without this repair the checkout is never imported.
# The real grader runs --sandbox docker where those functions DO execute. This moves the proxy toward
# Docker; it does NOT make it identical (dependency provisioning still differs).
import swegemma.harness.agent_runner as _ar
import swegemma.harness.verification as _vf
import swegemma.harness.container_setup as _cs

REQUIRED_BACKENDS = ["setuptools", "wheel", "editables"]
OPTIONAL_BACKENDS = ["flit_core", "hatchling", "poetry_core", "pdm_backend"]
COMP_WHEELS = str(DATA_DIR / "wheels")
_SUBPROC = {"SubprocessManager", "SubprocessSandbox"}
REPAIR_LOG = []
ACTIVE_RUN = None
ACTIVE_PHASE = "precondition"

# Idempotent: unwrap any previous patch before wrapping, so re-running this cell cannot nest wrappers.
_ORIG_INSTALL = getattr(_cs.install_editable_package, "__wrapped_original__", None) or _cs.install_editable_package


def _pip(docker, cid, pkg):
    """Run pip and KEEP the exit status; no `| tail` swallowing failures."""
    r = docker.exec(cid, "python3 -m pip install --no-index --find-links=/wheels "
                         "--find-links=" + COMP_WHEELS + " --no-build-isolation " + pkg)
    return {"rc": r.exit_code, "out": (r.stdout or ""), "err": (r.stderr or "")}


def patched_install_editable_package(docker, container_id):
    _ORIG_INSTALL(docker, container_id)
    if type(docker).__name__ not in _SUBPROC:
        return
    entry = {"container": str(container_id), "required": {}, "optional": {}, "phase": ACTIVE_PHASE, "run": ACTIVE_RUN}
    for b in REQUIRED_BACKENDS:
        entry["required"][b] = _pip(docker, container_id, b)
    for b in OPTIONAL_BACKENDS:
        entry["optional"][b] = _pip(docker, container_id, b)
    r = docker.exec(container_id, "cd /workspace && python3 -m pip install --no-index "
                                  "--find-links=/wheels --find-links=" + COMP_WHEELS +
                                  " --no-build-isolation --no-deps -e .")
    entry["editable"] = {"rc": r.exit_code, "out": (r.stdout or ""), "err": (r.stderr or "")}
    REPAIR_LOG.append(entry)
    (RESULTS/"setup_observations.json").write_text(json.dumps(REPAIR_LOG,indent=2),encoding="utf-8")
    failed_required = [b for b, v in entry["required"].items() if v["rc"] != 0]
    if failed_required:
        raise RuntimeError(f"required backend install failed: {failed_required}")
    if entry["editable"]["rc"] != 0:
        raise RuntimeError("editable install of /workspace failed rc="
                           + str(entry["editable"]["rc"]) + " :: " + entry["editable"]["err"][-300:])

    # Check imports inside the actual setup path in each agent/grading sandbox.
    pkg = (ACTIVE_RUN or {}).get('package')
    if not pkg:
        raise RuntimeError('Active task package is unavailable during setup')
    rw = docker.exec(container_id, 'cd /workspace && pwd -P')
    ws = (rw.stdout or '').strip()
    entry['workspace_real'] = ws
    for view, prefix in [('agent', ''), ('grading_flags', 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 ')]:
        suffix = '-s ' if view == 'grading_flags' else ''
        cmd = ('cd /workspace && ' + prefix + 'python3 ' + suffix + '-c "import ' + pkg +
               ' as m,sys; from pathlib import Path; print(Path(m.__file__).resolve()); print(sys.executable)"')
        result = docker.exec(container_id, cmd)
        lines = (result.stdout or '').strip().splitlines()
        entry[view] = {'rc': result.exit_code, 'path': lines[0] if lines else '',
                       'executable': lines[1] if len(lines)>1 else '', 'stderr': result.stderr}
    entry['provenance_ok'] = rw.exit_code == 0 and all(
        entry[v]['rc'] == 0 and under(entry[v]['path'], ws) for v in ('agent','grading_flags'))
    # Persist even a failing observation before raising; this is not an in-pytest hook.
    (RESULTS/'setup_observations.json').write_text(json.dumps(REPAIR_LOG,indent=2),encoding='utf-8')
    if not entry['provenance_ok']:
        raise RuntimeError('Actual sandbox import is outside its checkout: ' + json.dumps(entry))


patched_install_editable_package.__wrapped_original__ = _ORIG_INSTALL
for _m in (_ar, _vf, _cs):
    _m.install_editable_package = patched_install_editable_package
assert _ar.install_editable_package is _vf.install_editable_package is _cs.install_editable_package
print("repair bound on agent_runner, verification, container_setup (idempotent)")
print("NOTE: a pip success message alone is not proof; the preconditions below check what actually loads.")


In [ ]:
# PRECONDITION for EVERY selected task, using the patched setup path. CPU work only.
from swegemma.sandbox import SubprocessManager, sandbox_exec, sandbox_start, sandbox_stop
from swegemma.harness.container_setup import (
    extract_snapshot, install_test_dependencies, setup_baseline_commit,
    setup_container_wheels, setup_git_exclude, setup_workspace_test_config,
)
from swegemma.config import EvalConfig
import base64 as _b64mod
import hashlib as _hashlib
import tempfile as _tempfile
import re as _re_mod
import os
from swegemma.harness.verification import apply_patch_in_container
import traceback as _tbmod
from adk_submission import ModelRegistry

async def _precondition(task):
    global ACTIVE_RUN, ACTIVE_PHASE
    ACTIVE_RUN = {"task": task.instance_id, "candidate": None, "package": pkg_of(task)}
    ACTIVE_PHASE = "precondition"
    mgr = SubprocessManager(timeout_seconds=600)
    sandboxes_before = sandbox_set()
    sid = await sandbox_start(mgr)
    out = {"task": task.instance_id}
    try:
        cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / "snapshots",
                         results_dir=RESULTS / "precondition", submission_dir=WORKING_DIR,
                         models=ModelRegistry(), sandbox="subprocess", timeout_seconds=600,
                         display_mode="quiet")
        await asyncio.to_thread(setup_container_wheels, mgr, sid, cfg)
        await asyncio.to_thread(extract_snapshot, mgr, sid, snapshot_for(task))
        await asyncio.to_thread(setup_git_exclude, mgr, sid)
        await asyncio.to_thread(_cs.install_editable_package, mgr, sid)   # the patched one
        await asyncio.to_thread(install_test_dependencies, mgr, sid, task.repo, config=cfg)
        await asyncio.to_thread(setup_workspace_test_config, mgr, sid, repo=task.repo)
        await asyncio.to_thread(setup_baseline_commit, mgr, sid, "baseline")
        pkg = pkg_of(task)
        rw = await sandbox_exec(mgr, sid, "cd /workspace && pwd -P")
        out["ws"] = (rw.stdout or "").strip()
        ra = await sandbox_exec(mgr, sid,
                                'cd /workspace && python3 -c "import ' + pkg + ' as m,sys; '
                                'print(m.__file__); print(sys.executable)"')
        lines = (ra.stdout or "").strip().splitlines()
        out["agent_file"] = lines[0] if ra.exit_code == 0 and lines else ""
        out["agent_exe"] = lines[1] if ra.exit_code == 0 and len(lines) > 1 else ""
        rp = await sandbox_exec(mgr, sid, "cd /workspace && PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 "
                                          'python3 -s -c "import ' + pkg + ' as m,sys; '
                                          'print(m.__file__); print(sys.executable)"')
        lines = (rp.stdout or "").strip().splitlines()
        out["grading_file"] = lines[0] if rp.exit_code == 0 and lines else ""
        out["grading_exe"] = lines[1] if rp.exit_code == 0 and len(lines) > 1 else ""
    finally:
        try:
            await sandbox_stop(mgr, sid)
        except BaseException as stop_exc:
            out["teardown_error"] = f"{type(stop_exc).__name__}: {stop_exc}"
        # Teardown returning is not proof of teardown.
        out["cleanup_ok"], out["owned_sandboxes"] = confirm_cleanup(sandboxes_before)
        ACTIVE_RUN = None
    return out

PRECONDITIONS = []
for _t in SELECTED:
    _pc = run_sync(lambda _t=_t: _precondition(_t))
    ok = (under(_pc.get("agent_file", ""), _pc.get("ws", ""))
          and under(_pc.get("grading_file", ""), _pc.get("ws", ""))
          and not _pc.get("teardown_error")
          and _pc.get("cleanup_ok") is True)
    if _pc.get("teardown_error"):
        print("    TEARDOWN ERROR: " + str(_pc.get("teardown_error"))[:150])
    if not _pc.get("cleanup_ok"):
        print("    SANDBOX NOT CLEANED UP: " + str(_pc.get("owned_sandboxes")))
    _pc["ok"] = ok
    PRECONDITIONS.append(_pc)
    print(f"{_pc['task']:16s} ok={ok}")
    print(f"    workspace: {_pc.get('ws','')}")
    print(f"    agent    : {_pc.get('agent_file','') or '<import failed>'}")
    print(f"    grading  : {_pc.get('grading_file','') or '<import failed>'}")
(RESULTS / "preconditions.json").write_text(json.dumps(PRECONDITIONS, indent=2), encoding="utf-8")
bad = [p["task"] for p in PRECONDITIONS if not p["ok"]]
assert not bad, f"preconditions failed for {bad}; do not start the model"
print("ALL PRECONDITIONS PASSED")
print("Caveat: passing here does not guarantee the same resolution inside every later sandbox; the")
print("per-run setup observations verify imports again in the actual sandboxes. These are grading-flags probes, not direct in-pytest observations.")

CONTROL_PYTEST_CMD = {'rich_3278': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/test_ansi.py', 'rich_3535': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/test_cells.py', 'rich_3942': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/_card_render.py tests/test_markdown.py tests/test_markdown_no_hyperlinks.py'}
CONTROL_TARGET_NODES = {'rich_3278': ['test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'rich_3535': ['test_cells::test_is_single_cell_widths'], 'rich_3942': ['test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown_no_hyperlinks::test_markdown_render']}

# ---------------------------------------------------------------------------------------------
# Re-validate the four selected controls UNDER THIS NOTEBOOK'S SETUP.
#
# The saved controls came from the CPU screening kernel. Their environment_versions and
# setup_commands were recorded, but this session's versions are only observable here, at run time,
# so equivalence cannot be asserted in advance. Rather than assume it, the controls are re-run with
# the same patched setup path used above, and compared against what was recorded. This is CPU work
# and happens before the model server starts.
#
# The exact pytest command recorded by the screen is re-issued, so a difference in outcome cannot be
# blamed on a different invocation.
# ---------------------------------------------------------------------------------------------
import xml.etree.ElementTree as _ET

def _nodes_from_junit(xml_text):
    """Return (nodes, duplicates, parse_error).

    nodes is None when the report is missing, unparseable or contains no testcase elements. That
    stays distinct from an empty dict for the rest of the pipeline: an earlier version collapsed
    both to {} via `nodes or {}`, which let a missing report satisfy a reference arm whose expected
    failure set is empty. Duplicate node identities are reported rather than silently overwritten.
    """
    if not (xml_text or '').strip():
        return None, [], 'empty report'
    try:
        root = _ET.fromstring(xml_text)
    except Exception as exc:
        return None, [], f'unparseable report: {type(exc).__name__}: {exc}'
    out, dupes = {}, []
    seen_any = False
    for tc in root.iter('testcase'):
        seen_any = True
        cls = (tc.get('classname') or '').split('.')[-1]
        name = tc.get('name') or ''
        key = f'{cls}::{name}' if cls else name
        outcome = 'passed'
        if tc.find('failure') is not None:
            outcome = 'failed'
        elif tc.find('error') is not None:
            outcome = 'errored'
        elif tc.find('skipped') is not None:
            outcome = 'skipped'
        if key in out:
            dupes.append(key)
        out[key] = outcome
    if not seen_any:
        return None, [], 'report contains no testcase elements'
    return out, sorted(set(dupes)), None


def compare_control_nodes(observed, expected, targets, arm):
    """Full node/outcome comparison for a reproducibility control.

    Returns (ok, detail). `observed` may be None, which is always a failure. Every expected node
    must be present with the same outcome; every saved target node must fail in the baseline arm and
    pass in the reference arm. Comparing failure sets alone never proved the reference arm actually
    ran, let alone passed, the tests the baseline failed.
    """
    detail = {'missing': [], 'extra': [], 'changed': [], 'skipped': [],
              'targets_wrong': [], 'n_observed': 0, 'n_expected': len(expected)}
    if observed is None:
        detail['reason'] = 'no usable JUnit report'
        return False, detail
    if not observed:
        detail['reason'] = 'no nodes collected'
        return False, detail
    detail['n_observed'] = len(observed)
    detail['missing'] = sorted(set(expected) - set(observed))
    detail['extra'] = sorted(set(observed) - set(expected))
    detail['changed'] = sorted(f'{n}: expected {expected[n]}, observed {observed[n]}'
                               for n in set(expected) & set(observed)
                               if expected[n] != observed[n])
    detail['skipped'] = sorted(n for n, o in observed.items() if o == 'skipped')
    want_target = 'failed' if arm == 'baseline' else 'passed'
    detail['targets_wrong'] = sorted(
        f'{n}: expected {want_target}, observed {observed.get(n, "ABSENT")}'
        for n in targets if observed.get(n) != want_target)
    ok = not (detail['missing'] or detail['extra'] or detail['changed'] or detail['targets_wrong'])
    if ok:
        detail['reason'] = ''
    else:
        detail['reason'] = 'node map does not match the saved control'
    return ok, detail


async def _revalidate_control(task, arm):
    """Run one control arm in a fresh sandbox under this notebook's setup. arm: baseline|reference."""
    global ACTIVE_RUN, ACTIVE_PHASE
    ACTIVE_RUN = {"task": task.instance_id, "candidate": None, "package": pkg_of(task)}
    ACTIVE_PHASE = "control_" + arm
    mgr = SubprocessManager(timeout_seconds=900)
    # Snapshot BEFORE this arm creates anything, so only its own sandboxes are attributable to it.
    sandboxes_before = sandbox_set()
    sid = await sandbox_start(mgr)
    rec = {"task": task.instance_id, "arm": arm}
    try:
        cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / "snapshots",
                         results_dir=RESULTS / "control", submission_dir=WORKING_DIR,
                         models=ModelRegistry(), sandbox="subprocess", timeout_seconds=900,
                         display_mode="quiet")
        await asyncio.to_thread(setup_container_wheels, mgr, sid, cfg)
        await asyncio.to_thread(extract_snapshot, mgr, sid, snapshot_for(task))
        await asyncio.to_thread(setup_git_exclude, mgr, sid)
        await asyncio.to_thread(_cs.install_editable_package, mgr, sid)
        await asyncio.to_thread(install_test_dependencies, mgr, sid, task.repo, config=cfg)
        await asyncio.to_thread(setup_workspace_test_config, mgr, sid, repo=task.repo)
        await asyncio.to_thread(setup_baseline_commit, mgr, sid, "baseline")

        # PATCH APPLICATION.
        #
        # Comparison version 1 aborted here: all eight test-patch and all four reference-patch
        # applications returned 128. Measured with real git: 128 means the patch file was MISSING
        # or EMPTY ("can't open patch" / "No valid patches in input"); a patch that merely fails to
        # apply returns 1. The old helper wrote the patch with a shell `python3 -c` one-liner and
        # never checked that command's exit code, so a failed write was invisible and every pytest
        # run then passed against an unpatched checkout.
        #
        # This now does exactly what the screening run did for 38 tasks without a single failure:
        # write to a HOST file, copy it in with mgr.copy_to, and apply with the harness's own
        # multi-strategy apply_patch_in_container. Creation is verified before application, and the
        # command output is preserved either way.
        async def _put(text, name):
            with _tempfile.NamedTemporaryFile("w", suffix="_" + name, delete=False,
                                              encoding="utf-8") as f:
                f.write(text if text.endswith("\n") else text + "\n")
                host = Path(f.name)
            try:
                await asyncio.to_thread(mgr.copy_to, sid, host, "/tmp/")
                return "/tmp/" + host.name, len(text.encode("utf-8")), \
                    _hashlib.sha256(text.encode("utf-8")).hexdigest()
            finally:
                os.unlink(host)

        async def _apply(diff, label):
            """Return (rc, evidence). rc None means no patch was requested for this arm."""
            ev = {"label": label, "requested": bool((diff or "").strip())}
            if not ev["requested"]:
                ev["note"] = "no patch requested for this arm (not a successful application)"
                return None, ev
            path, size, sha = await _put(diff, label + ".patch")
            ev.update({"sandbox_path": path, "bytes": size, "sha256": sha})
            # Verify creation BEFORE attempting application, so a missing or empty file is named
            # as such instead of surfacing later as an opaque git exit 128.
            probe = await sandbox_exec(mgr, sid, "wc -c < " + path + " || echo MISSING")
            ev["created_bytes"] = (probe.stdout or "").strip()
            ev["create_probe_rc"] = probe.exit_code
            if probe.exit_code != 0 or not (ev["created_bytes"] or "").strip().isdigit() \
                    or int(ev["created_bytes"]) == 0:
                ev["error"] = "patch file was not created, or is empty; not attempting application"
                return 128, ev
            code, out, err = await asyncio.to_thread(apply_patch_in_container, mgr, sid, path)
            ev.update({"apply_rc": code, "stdout": (out or "")[-1500:], "stderr": (err or "")[-1500:]})
            return code, ev

        # Mirror the screening run: restore any test path before re-applying the verification patch.
        for _tf in _re_mod.findall(r"^\+\+\+ b/(\S+)", getattr(task, "test_patch", "") or "", _re_mod.M):
            await sandbox_exec(mgr, sid, "cd /workspace && git checkout HEAD -- " + _tf)
            await sandbox_exec(mgr, sid, "cd /workspace && git clean -f -- " + _tf)

        rec["test_patch_rc"], rec["test_patch_evidence"] = await _apply(
            getattr(task, "test_patch", ""), "testpatch")
        if arm == "reference":
            rec["patch_rc"], rec["patch_evidence"] = await _apply(getattr(task, "patch", ""), "gold")
        else:
            # NOT 0. The baseline arm applies no source patch; reporting 0 made "nothing was done"
            # indistinguishable from "applied successfully", which is how version 1 read as clean.
            rec["patch_rc"], rec["patch_evidence"] = None, {
                "label": "gold", "requested": False,
                "note": "baseline arm applies no source patch by design"}

        cmd = CONTROL_PYTEST_CMD[task.instance_id]
        xml = "/tmp/ctl_" + arm + ".xml"
        r = await sandbox_exec(mgr, sid, "cd /workspace && " + cmd + " -q --junit-xml=" + xml)
        rec["pytest_exit"] = r.exit_code
        rec["pytest_cmd"] = cmd
        rec["pytest_stdout"] = (r.stdout or "")[-8000:]
        rec["pytest_stderr"] = (getattr(r, "stderr", "") or "")[-4000:]
        rx = await sandbox_exec(mgr, sid, "cat " + xml)
        # The raw report is preserved verbatim. It is never synthesised: if cat failed, that is
        # recorded as a missing report, not replaced with an empty document.
        rec["junit_xml"] = (rx.stdout or "") if rx.exit_code == 0 else None
        rec["junit_read_rc"] = rx.exit_code
        nodes, dupes, parse_error = _nodes_from_junit(rec["junit_xml"] or "")
        rec["nodes"] = nodes
        rec["duplicate_nodes"] = dupes
        rec["junit_parse_error"] = parse_error if rx.exit_code == 0 else "JUnit report not readable"
        rv = await sandbox_exec(mgr, sid,
            "python3 -c \"import importlib.metadata as m;"
            "print({p: m.version(p) for p in ['swegemma','adk-submission','adk-eval-core','google-adk']})\"")
        rec["environment_versions"] = (rv.stdout or "").strip()
    except BaseException as exc:
        rec["exception"] = f"{type(exc).__name__}: {exc}"
        rec["traceback"] = _tbmod.format_exc()
        rec.setdefault("nodes", None)
    finally:
        try:
            await sandbox_stop(mgr, sid)
        except BaseException as stop_exc:
            rec["teardown_error"] = f"{type(stop_exc).__name__}: {stop_exc}"
            rec["teardown_traceback"] = _tbmod.format_exc()
        # Teardown that RETURNS is not proof of teardown. Verify the arm's own sandboxes are gone,
        # with the same bounded grace the dispatch loop uses. Nothing is deleted here, and a
        # sandbox that existed before this arm is never attributed to it.
        rec["cleanup_ok"], rec["owned_sandboxes"] = confirm_cleanup(sandboxes_before)
        ACTIVE_RUN = None
    return rec


CONTROL_DIR = RESULTS / "control_evidence"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_RECHECK = []


def _persist_arm(record):
    """Write one arm's evidence the moment it exists, so a later failure cannot erase it."""
    stem = f"{record['task']}__{record['arm']}"
    raw = record.pop("junit_xml", None)
    if raw is not None:
        (CONTROL_DIR / (stem + ".junit.xml")).write_text(raw, encoding="utf-8")
    else:
        (CONTROL_DIR / (stem + ".NO_JUNIT.txt")).write_text(
            "no readable JUnit report; nothing was synthesised\n"
            + str(record.get("junit_parse_error") or ""), encoding="utf-8")
    (CONTROL_DIR / (stem + ".json")).write_text(
        json.dumps(record, indent=2, default=str), encoding="utf-8")


CONTROL_STOP_REASON = None
for _t in SELECTED:
    if CONTROL_STOP_REASON:
        break
    saved = CONTROL_EVIDENCE[_t.instance_id]
    targets = CONTROL_TARGET_NODES[_t.instance_id]
    for _arm, _phase in (("baseline", "negative"), ("reference", "positive")):
        try:
            got = run_sync(lambda _t=_t, _a=_arm: _revalidate_control(_t, _a))
        except BaseException as _exc:
            got = {"task": _t.instance_id, "arm": _arm, "nodes": None,
                   "exception": f"{type(_exc).__name__}: {_exc}",
                   "traceback": _tbmod.format_exc(),
                   "cleanup_ok": False,
                   "owned_sandboxes": ["unknown: the arm did not reach its cleanup check"]}
        want = saved[_phase]
        expected = dict(want.get("nodes") or {})
        ok_nodes, detail = compare_control_nodes(got.get("nodes"), expected, targets, _arm)
        # A sandbox this arm leaked would be snapshotted as "pre-existing" by the dispatch loop's
        # own cleanup check, so it would never be caught there. It has to be caught here.
        agree = (ok_nodes
                 and not got.get("exception")
                 and not got.get("teardown_error")
                 and got.get("cleanup_ok") is True
                 and not got.get("duplicate_nodes")
                 and got.get("pytest_exit") == want.get("pytest_exit")
                 # baseline applies no source patch, so None is correct there and 0 would
                 # be a false success; reference MUST have applied its patch.
                 and got.get("patch_rc") == (None if _arm == "baseline" else 0)
                 and got.get("test_patch_rc") == 0
                 and not [n for n, o in (got.get("nodes") or {}).items() if o == "errored"])
        got["agrees_with_saved"] = bool(agree)
        got["saved_exit"] = want.get("pytest_exit")
        got["node_comparison"] = detail
        got["n_targets"] = len(targets)
        CONTROL_RECHECK.append(got)
        _persist_arm(dict(got))          # immediately, before the next arm runs
        print(f"  {_t.instance_id:14s} {_arm:9s} exit={got.get('pytest_exit')} "
              f"(saved {want.get('pytest_exit')}) nodes={detail['n_observed']}/{detail['n_expected']} "
              f"targets={len(targets)} agrees={agree}")
        for _k in ("missing", "extra", "changed", "skipped", "targets_wrong"):
            if detail.get(_k):
                print(f"                 {_k}: {detail[_k][:4]}"
                      + (" ..." if len(detail[_k]) > 4 else ""))
        if got.get("duplicate_nodes"):
            print(f"                 DUPLICATE node identities: {got['duplicate_nodes'][:4]}")
        if got.get("junit_parse_error"):
            print(f"                 junit: {got['junit_parse_error']}")
        if got.get("exception"):
            print(f"                 EXCEPTION: {got['exception'][:150]}")
        if got.get("teardown_error"):
            print(f"                 TEARDOWN ERROR: {got['teardown_error'][:150]}")
        if not got.get("cleanup_ok"):
            print(f"                 SANDBOX NOT CLEANED UP: {got.get('owned_sandboxes')}")
        # Evidence for this arm is already on disk (_persist_arm above). Stop before any further
        # arm runs, so a leaked sandbox cannot be inherited as "pre-existing" by the next one.
        if got.get("teardown_error") or not got.get("cleanup_ok"):
            CONTROL_STOP_REASON = (
                f"{_t.instance_id}/{_arm}: control sandbox cleanup could not be confirmed"
                + (f" ({got['teardown_error']})" if got.get("teardown_error") else ""))
            print("Stopping control re-validation:", CONTROL_STOP_REASON)
            break

(RESULTS / "control_recheck.json").write_text(
    json.dumps(CONTROL_RECHECK, indent=2, default=str), encoding="utf-8")
assert not CONTROL_STOP_REASON, (
    "control sandbox cleanup failed: " + str(CONTROL_STOP_REASON) +
    "; evidence is preserved in control_evidence/, and the model is NOT started")
_expected_arms = 2 * len(SELECTED)
assert len(CONTROL_RECHECK) == _expected_arms, (
    f"only {len(CONTROL_RECHECK)} of {_expected_arms} control arms ran; do not start the model")
_bad = [f"{c['task']}/{c['arm']}" for c in CONTROL_RECHECK if not c["agrees_with_saved"]]
assert not _bad, ("controls do not reproduce under this notebook's setup: " + str(_bad) +
                  "; do not start the model")
print("ALL FOUR CONTROLS REPRODUCE UNDER THIS SETUP (both arms, full node map, targets verified)")


In [ ]:
# vLLM with the public harness serving configuration; private parity unverified. Started only after preconditions pass.
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
TARGET_MODEL_NAME = "gemma-4-31b-it-qat-w4a16-ct"
tp_size = 4 if N_GPU >= 4 else (2 if N_GPU >= 2 else 1)
adapters = discover_adapters(str(CAND_DIRS["OFF"]), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
print("adapters discovered:", getattr(adapters, "adapters", {}) or "none (expected)")
server_instance = None
models = None
SERVER_STARTUP_S = None
SERVER_STOPPED = False
if not DISPATCH_CONFIRM:
    print('Dispatch disabled: vLLM is not instantiated or started.')
else:
    assert N_GPU == 4 and all('L4' in n for n in GPU_NAMES), 'Pilot requires 4x L4'
    assert elapsed_min() + 20 + RUN_RESERVE_MIN <= SESSION_CAP_MIN, 'Insufficient allowance for startup and one run'
    for tid in TASK_IDS:
        for cand in CAND_DIRS:
            target = RESULTS / f'{cand}__{tid}'
            if target.exists() and any(target.iterdir()):
                raise RuntimeError(f'Existing artifacts: {target}')
    try:
        SERVER_T0 = time.time()
        vllm_cfg = VllmConfig(
            model=str(MODEL_PATH), port=8000, host="127.0.0.1",
            tool_call_parser="gemma4", reasoning_parser="gemma4",
            default_chat_template_kwargs={"enable_thinking": True},
            max_model_len=32768,
            dtype="bfloat16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "auto",
            gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
            enable_lora=True, max_loras=8, max_lora_rank=128,
            tensor_parallel_size=tp_size, startup_timeout=60 * 20)
        server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
        server_instance.start()
        SERVER_STARTUP_S = time.time() - SERVER_T0
        print(f"vLLM up at {server_instance.base_url} (tp={tp_size}) in {SERVER_STARTUP_S/60:.1f} min")
        print("NOTE: the server default enable_thinking=True is overridden per-request by each candidate's")
        print("extra_body.chat_template_kwargs, OFF disables thinking; ON requests a 1024-token thinking budget.")
        models = server_instance.create_model_registry(
            aliases=[TARGET_MODEL_NAME], model_prefix="openai/", api_key="EMPTY")

    except BaseException:
        if server_instance is not None:
            server_instance.stop()
            SERVER_STOPPED = True
        raise


In [ ]:
# Evaluator imports these functions by name: instrument its bindings, preserving signatures.
import functools, inspect, time, traceback
import swegemma.evaluate as _ev
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig

PHASE = {}

def instrument_phase(fn, label):
    original = getattr(fn, '__pilot_original__', fn)
    @functools.wraps(original)
    async def wrapped(*a, **k):
        global ACTIVE_PHASE
        previous = ACTIVE_PHASE
        ACTIVE_PHASE = label
        started = time.perf_counter()
        context = k.get('context')
        try:
            return await original(*a, **k)
        finally:
            PHASE.setdefault(label + '_phase_s', []).append(time.perf_counter() - started)
            if label == 'agent' and context is not None and getattr(context, 'agent_start_time', None) is not None:
                PHASE.setdefault('agent_loop_s', []).append(context.agent_elapsed_seconds)
            ACTIVE_PHASE = previous
    wrapped.__pilot_original__ = original
    return wrapped

_ev.run_agent_sandbox = _ar.run_agent_sandbox = instrument_phase(_ev.run_agent_sandbox, 'agent')
_ev.verify_task = _vf.verify_task = instrument_phase(_ev.verify_task, 'grading')


# --------------------------------------------------------------------------------------------
# Outcome classification.
#
# swegemma's Evaluator.run() RETURNS NORMALLY in most failure modes and records the reason in
# TaskResult.error (models/task.py:181), which is persisted to results_dir/task_results.jsonl.
# Reading only the escaped exception therefore misses the common case. Both are consumed below.
#
# Persisted error vocabulary, read from the installed harness source, not guessed:
#   'Agent exceeded session timeout (N min)'                     -> candidate (budget)
#   'Failed to apply agent patch: ...'                           -> candidate (its own patch)
#   'Failed to apply test_patch: ...'                            -> environment (answer key)
#   'Missing JUnit XML report (possible premature os._exit(0))'  -> grading not observed
#   'Pytest stdout summary indicates zero or no passing tests'   -> graded, zero passes (a RESULT)
#   'Sandbox execution error: ...'                               -> environment
#   'Evaluation error: ...' / 'Unexpected evaluation worker error: ...' -> wrapper, inspect cause
#
# The harness contains NO BadRequestError or context-window handling of its own (grep over the
# installed source finds none), so any such text comes from litellm or the model server. A 400 is a
# candidate failure only when its cause is the request the agent produced; a 400 from an unready or
# misconfigured server is an environment failure. An unexamined 400 is therefore NOT attributed to
# the candidate.
# --------------------------------------------------------------------------------------------
_CANDIDATE_MARKERS = (
    'contextwindowexceeded', 'context window', 'maximum context length',
    'reduce the length of the messages', 'too many tokens', 'prompt is too long',
    'exceeded session timeout', 'turns budget', 'tool call budget',
    'failed to apply agent patch',
)
# Specific, self-describing environment causes. NOT wrappers.
_ENVIRONMENT_MARKERS = (
    'failed to apply test_patch', 'connection refused', 'connection error',
    'apiconnectionerror', 'read timed out', 'service unavailable', 'internal server error',
    'bad gateway', 'out of memory', 'cuda error', 'no such file or directory',
)
# Generic wrappers: the harness puts the REAL cause inside these, so they are examined only after
# the specific markers above have had a chance to match the text they contain.
_WRAPPER_MARKERS = (
    'unexpected evaluation worker error', 'sandbox execution error', 'evaluation error',
)
_UNOBSERVED_MARKERS = ('missing junit xml report',)
# A graded run with zero passes is an OUTCOME, not a failure of the run.
_RESULT_MARKERS = ('indicates zero or no passing tests',)
# The agent loop ended with no patch submitted. Observed in compare kernel version 2 (candidate B on
# rich_3278): the model emitted `<|tool_call>...<tool_call|>` with mismatched delimiters, the server's
# tool parser did not recognise it as a tool call, the harness treated the text as a final answer,
# and swegemma persisted exactly this message. Nothing about the environment failed: four L4s were
# up, the sandbox was clean, both setup provenance probes passed, and the SAME server had just
# carried candidate A through 60 turns. This is the candidate's own tool use, so it is a candidate
# outcome and must not stop the other candidate's runs.
#
# Deliberately checked AFTER _ENVIRONMENT_MARKERS: when a real environment cause is named in the
# same string, that cause wins. A missing submission is what remains when nothing else is wrong.
_CANDIDATE_NO_SUBMISSION_MARKERS = ('completed execution without calling submit_patch',)

# Worst-first. A genuine environment failure must never be hidden behind a candidate error, and a
# candidate error must never be hidden behind a clean-looking second source.
_PRECEDENCE = ('environment', 'unobserved_grading', 'candidate', 'ok')


def _classify_one(text):
    """Classify ONE error string. Candidate causes are examined before generic wrappers."""
    low = text.lower()
    for m in _RESULT_MARKERS:
        if m in low:
            return 'ok', m
    for m in _UNOBSERVED_MARKERS:
        if m in low:
            return 'unobserved_grading', m
    # Before the wrappers: 'Unexpected evaluation worker error: ...ContextWindowExceededError...'
    # is a context-window failure reported through a wrapper, and belongs to the candidate.
    for m in _CANDIDATE_MARKERS:
        if m in low:
            return 'candidate', m
    for m in _ENVIRONMENT_MARKERS:
        if m in low:
            return 'environment', m
    # After the specific environment causes, before the generic wrappers: a run that simply never
    # submitted has an identified cause, and that cause is the candidate's.
    for m in _CANDIDATE_NO_SUBMISSION_MARKERS:
        if m in low:
            return 'candidate', 'agent ended without submitting a patch (no environment cause found)'
    if 'badrequesterror' in low or 'invalidrequesterror' in low:
        return 'environment', 'BadRequest/InvalidRequest with no identified cause (not charged to the candidate)'
    for m in _WRAPPER_MARKERS:
        if m in low:
            return 'environment', 'wrapper with no identified cause: ' + m
    return 'environment', 'unclassified error'


def classify_outcome(escaped_error, persisted_error):
    """Return (outcome, reason).

    outcome is one of: ok | candidate | environment | unobserved_grading.

    BOTH sources are classified before any decision is taken. An earlier implementation returned on
    the first source that matched, so an escaped candidate error silently discarded a persisted
    environment failure. Results are combined worst-first via _PRECEDENCE, and the reason names
    every source so a combined verdict can be audited.
    """
    per_source = []
    for label, text in (('escaped', escaped_error), ('persisted', persisted_error)):
        if text:
            cls, why = _classify_one(text)
            per_source.append((label, cls, why))
    if not per_source:
        return 'ok', ''
    for want in _PRECEDENCE:
        hits = [p for p in per_source if p[1] == want]
        if hits:
            label, cls, why = hits[0]
            others = '; '.join(f'{l}={c}' for l, c, _ in per_source if l != label)
            return cls, f'{label}: {why}' + (f' [also {others}]' if others else '')
    return 'ok', ''


def read_persisted_result(out_dir, tid):
    """(record_or_None, persisted_error, n_records) from the evaluator's own output file."""
    jf = Path(out_dir) / 'task_results.jsonl'
    if not jf.exists():
        return None, None, 0
    recs = [json.loads(l) for l in jf.read_text(encoding='utf-8').splitlines() if l.strip()]
    if not recs:
        return None, None, 0
    rec = recs[0]
    if rec.get('instance_id') not in (None, tid):
        return rec, f"result is for {rec.get('instance_id')}, expected {tid}", len(recs)
    return rec, rec.get('error'), len(recs)



ORDER = [('rich_3278', 'OFF'), ('rich_3278', 'ON'), ('rich_3535', 'ON'), ('rich_3535', 'OFF'), ('rich_3942', 'OFF'), ('rich_3942', 'ON')]
RUNS = []
STOP_REASON = None
print('planned run order:', ORDER)
print('backend for BOTH preconditions and evaluation: subprocess')
try:
    if not DISPATCH_CONFIRM:
        print('DISPATCH_CONFIRM=False: no server was started and no model run is dispatched.')
    else:
        for tid, cand in ORDER:
            # Admission limit, not a guarantee that in-flight OS processes stop at this deadline.
            if elapsed_min() + RUN_RESERVE_MIN > SESSION_CAP_MIN:
                STOP_REASON = 'insufficient session allowance for another run'
                print(STOP_REASON + '; preserving partial artifacts.')
                break
            out = RESULTS / f'{cand}__{tid}'
            if out.exists() and any(out.iterdir()):
                raise RuntimeError(f'Existing results at {out}; refusing to mix runs')
            out.mkdir(parents=True, exist_ok=True)
            sandboxes_before = sandbox_set()
            PHASE.clear()
            ACTIVE_RUN = {'task': tid, 'candidate': cand, 'package': pkg_of(ALL_BY_ID[tid])}
            first_repair = len(REPAIR_LOG)
            cfg = EvalConfig(
                tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR/'snapshots', results_dir=out,
                submission_dir=CAND_DIRS[cand], models=models, sandbox='subprocess',
                graph_dir=str(DATA_DIR/'graphs'), embeddings_dir=str(DATA_DIR/'embeddings'),
                task_ids=[tid], limits=LIMITS, generation_constraints=GEN_CONSTRAINTS,
                adapter_manifest=adapters, concurrency=1, display_mode='quiet',
                context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
                events_compaction_config=EventsCompactionConfig(
                    compaction_interval=15, overlap_size=2, token_threshold=20480, event_retention_size=5),
                **BUDGET)
            started = time.perf_counter()
            err = None
            try:
                run_sync(_ev.Evaluator(cfg).run)
            except Exception:
                err = traceback.format_exc()
                print(err[-1200:])
            finally:
                wall = time.perf_counter() - started
                repairs = REPAIR_LOG[first_repair:]
                # Per-phase provenance is three-valued. A phase that never ran is NOT a failure:
                # in pilot v1 candidate B raised before grading started, so the grading probe never
                # executed, and treating that absence as a provenance failure stopped two unrelated runs.
                seen = {r.get('phase'): bool(r.get('provenance_ok')) for r in repairs}
                phase_status = {ph: ('passed' if seen.get(ph) else
                                     'failed' if ph in seen else 'not_attempted')
                                for ph in ('agent', 'grading')}
                provenance_failed = any(v == 'failed' for v in phase_status.values())
                # Both sources: the escaped exception AND what the evaluator persisted. The
                # evaluator returns normally in most failure modes, so `err` alone is usually None.
                rec, persisted_err, n_recs = read_persisted_result(out, tid)
                outcome, outcome_reason = classify_outcome(err, persisted_err)
                grading_observed = bool(PHASE.get('grading_phase_s'))
                if outcome == 'ok' and n_recs == 0:
                    outcome, outcome_reason = 'environment', 'evaluator wrote no task result'
                elif n_recs > 1:
                    outcome, outcome_reason = 'environment', f'{n_recs} task results written, expected 1'
                elif outcome == 'ok' and not grading_observed:
                    # Grading that never started is NOT a provenance failure and NOT a candidate
                    # failure: it is an absence of observation, and is reported as one.
                    outcome, outcome_reason = 'unobserved_grading', 'grading phase never ran'
                candidate_error = outcome == 'candidate'
                environment_error = outcome == 'environment'
                unobserved_grading = outcome == 'unobserved_grading'
                cleanup_ok, owned_sandboxes = confirm_cleanup(sandboxes_before)
                RUNS.append(dict(task=tid, candidate=cand, dir=str(out), wall_s=round(wall,3),
                    agent_phase_s=list(PHASE.get('agent_phase_s', [])),
                    grading_phase_s=list(PHASE.get('grading_phase_s', [])),
                    agent_loop_s=list(PHASE.get('agent_loop_s', [])),
                    setup_provenance=repairs,
                    phase_status=phase_status,
                    provenance_failed=provenance_failed,
                    candidate_error=candidate_error,
                    environment_error=environment_error,
                    unobserved_grading=unobserved_grading,
                    outcome=outcome, outcome_reason=outcome_reason,
                    grading_observed=grading_observed,
                    persisted_error=persisted_err,
                    n_persisted_results=n_recs,
                    cleanup_ok=cleanup_ok, owned_sandboxes=owned_sandboxes,
                    both_setup_imports_verified=all(v == 'passed' for v in phase_status.values()),
                    error=err, elapsed_min_at_start=elapsed_min()-wall/60))
                (RESULTS/'runs.json').write_text(json.dumps(RUNS,indent=2),encoding='utf-8')
                ACTIVE_RUN = None
            print(cand, tid, 'wall_s=', round(wall,3), 'phase_s=', PHASE)
            last = RUNS[-1]
            print('  phase status:', last['phase_status'])
            print('  outcome:', last['outcome'], '|', last['outcome_reason'] or 'clean')
            print('  grading observed:', last['grading_observed'],
                  '| results written:', last['n_persisted_results'])
            print('  cleanup of THIS run\'s sandboxes:', last['cleanup_ok'],
                  '' if last['cleanup_ok'] else f"| still present: {last['owned_sandboxes']}")
            if not last['cleanup_ok']:
                # A leaked sandbox means the next run does not start from a known state.
                STOP_REASON = 'sandbox cleanup for this run could not be confirmed'
                print('Stopping further dispatch:', STOP_REASON)
                break
            if last['unobserved_grading']:
                STOP_REASON = 'grading was not observed; the result cannot be graded or attributed'
                print('Stopping further dispatch:', STOP_REASON)
                break
            if last['provenance_failed'] or last['environment_error']:
                STOP_REASON = ('provenance failure' if last['provenance_failed']
                               else f"environment failure: {last['outcome_reason']}")
                print('Stopping further dispatch:', STOP_REASON)
                break
            if last['candidate_error']:
                # Candidate-only failure. Continue ONLY if the server is still healthy and the next
                # run will get a clean sandbox; otherwise stop. Never continue blindly.
                healthy = False
                try:
                    import urllib.request
                    with urllib.request.urlopen(server_instance.base_url.rstrip('/') + '/models',
                                                timeout=20) as resp:
                        healthy = resp.status == 200
                except Exception as probe_err:
                    print('  model-server probe failed:', repr(probe_err)[:160])
                leftovers = sorted(Path('/tmp').glob('swegemma_sandbox_*'))
                print('  server healthy:', healthy, '| leftover sandboxes:', len(leftovers))
                if not healthy:
                    STOP_REASON = 'candidate error and the model server is not healthy'
                    print('Stopping further dispatch:', STOP_REASON)
                    break
                print('Continuing: candidate-only error, server healthy, next run gets a fresh sandbox.')
finally:
    if server_instance is not None:
        server_instance.stop()
        SERVER_STOPPED = True


In [ ]:
# Result packet: per-run failure attribution derived from structured traces.
# Anything not observable is reported as "unavailable", never as 0.
import csv, json, re
from pathlib import Path

UNAVAILABLE = "unavailable"

def load_trace(d, tid):
    p = d / "traces" / ("trace_" + tid.replace("/", "__") + ".json")
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return None

TRACE_STAT_KEYS = ("tool_calls", "edit_calls", "shell_edit_hints",
                   "repeated_identical_cmds", "tool_errors",
                   "first_source_edit_attempt_step", "finish_reasons",
                   "prompt_tokens", "completion_tokens",
                   "successful_tool_calls", "rejected_tool_calls",
                   "acknowledged_source_edit_calls", "tmp_scratch_writes",
                   "repeated_identical_tool_calls", "unparsed_tool_call_texts",
                   "tool_use_flags")

# Scratch reproduction files. The system prompt tells the agent to keep them in /tmp, and
# write_file/edit_file refuse /tmp, so they arrive as shell heredocs. Counting them as source edits
# made compare kernel version 2 credit candidate A with 54 "shell-edit workarounds" on a run that
# touched no repository file at all.
_SCRATCH_DIRS = ("/tmp/", "/var/tmp/", "/dev/")
_SCRATCH_NAMES = {"repro.py", "repro2.py", "debug.py", "scratch.py", "test_repro.py", "check.py"}

_REDIRECT_TARGET = re.compile(r">>?\s*(?P<p>[^\s;|&<>()]+)")
_TEE_TARGET = re.compile(r"\btee\b(?:\s+-\S+)*\s+(?P<p>[^\s;|&<>()]+)")
_INPLACE_EDIT = re.compile(r"(^|[\s;|&])(sed\s+-i|patch\b|python3?\s+-c\s+.{0,40}(write|replace))")
_PY_TOKEN = re.compile(r"[^\s;|&<>()]+\.py\b")

# The tool rejected the call before running it: the arguments never formed a usable request. This
# is the shape seen 42 times in compare kernel version 2, where the payload was split on its own
# content and old_string was lost.
_REJECTION_MARKERS = ("mandatory input parameters are not present",
                      "unknown tool", "no such tool", "unexpected keyword argument",
                      "missing required", "invalid arguments")


def _is_scratch_path(p):
    return p.startswith(_SCRATCH_DIRS) or p.rsplit("/", 1)[-1] in _SCRATCH_NAMES


def shell_write_targets(cmd):
    """(source_targets, scratch_targets) a shell command names as write destinations.

    Only paths ending in .py count, because a repository source edit is what this metric claims to
    measure. `cat > /tmp/repro.py <<EOF` yields a scratch target and no source target.
    """
    cand = [m.group("p") for m in _REDIRECT_TARGET.finditer(cmd)]
    cand += [m.group("p") for m in _TEE_TARGET.finditer(cmd)]
    if _INPLACE_EDIT.search(cmd):
        cand += _PY_TOKEN.findall(cmd)
    src, scratch = [], []
    for p in cand:
        if not p.endswith(".py"):
            continue
        (scratch if _is_scratch_path(p) else src).append(p)
    return src, scratch


def observation_status(step):
    """('ok'|'rejected'|'error'|'none', text) for one step's tool observation.

    swegemma reports tool failures in two different shapes and the previous implementation saw only
    one of them. A rejected call carries a bare {"error": ...} with no `status` key, so counting
    `status == "error"` alone reported 1 error for a run that had 43.
    """
    obs = step.get("observation")
    if not obs:
        return "none", ""
    content = obs.get("content", "")
    parsed = None
    if isinstance(content, dict):
        parsed = content
    elif isinstance(content, str):
        try:
            parsed = json.loads(content)
        except (ValueError, TypeError):
            parsed = None
    if not isinstance(parsed, dict):
        return ("ok", "") if content else ("none", "")
    text = str(parsed.get("error") or parsed.get("error_message") or "")
    is_error = ("error" in parsed or parsed.get("status") == "error"
                or (isinstance(parsed.get("exit_code"), int) and parsed["exit_code"] != 0))
    if not is_error:
        return "ok", ""
    low = text.lower()
    if any(m in low for m in _REJECTION_MARKERS):
        return "rejected", text
    return "error", text


def trace_stats(tr):
    if tr is None:
        return {k: UNAVAILABLE for k in TRACE_STAT_KEYS}
    cmds, tools, calls, finish = [], [], [], []
    errors = rejected = ok_calls = 0
    edit_calls = shell_edits = scratch_writes = ack_edits = unparsed_texts = 0
    first_edit = None
    for step in tr.get("steps", []):
        status, _text = observation_status(step)
        if status == "rejected":
            rejected += 1
            errors += 1
        elif status == "error":
            errors += 1
        elif status == "ok":
            ok_calls += 1
        step_calls = step.get("tool_calls") or []
        for tc in step_calls:
            fn = tc.get("function_name", "")
            tools.append(fn)
            args = tc.get("arguments") or {}
            calls.append(json.dumps([fn, args], sort_keys=True, default=str))
            if fn in ("edit_file", "write_file"):
                edit_calls += 1
                fp = str(args.get("filepath", ""))
                if fp.endswith(".py") and not _is_scratch_path(fp):
                    if first_edit is None:
                        first_edit = step.get("step_id")
                    # The tool accepted the call. Whether the file's bytes changed is NOT
                    # measured here: no before/after comparison is made anywhere in this trace.
                    if status == "ok":
                        ack_edits += 1
            if fn == "run_command":
                c = str(args.get("command", ""))
                cmds.append(c)
                src, scratch = shell_write_targets(c)
                if scratch:
                    scratch_writes += 1
                if src:
                    shell_edits += 1
                    if first_edit is None:
                        first_edit = step.get("step_id")
                    if status == "ok":
                        ack_edits += 1
        # An agent turn that produced text but no parsed tool call, where the text still looks like
        # an attempted call. This is how candidate B's run ended in compare kernel version 2.
        if not step_calls and step.get("source") == "agent":
            msg = str(step.get("message") or "")
            if "tool_call" in msg or "tool_code" in msg:
                unparsed_texts += 1
        fr = step.get("finish_reason") or (step.get("extra") or {}).get("finish_reason")
        if fr:
            finish.append(str(fr))
    rep = sum(1 for i in range(1, len(cmds)) if cmds[i] == cmds[i - 1])
    rep_tools = sum(1 for i in range(1, len(calls)) if calls[i] == calls[i - 1])
    fm = tr.get("final_metrics") or {}
    return {"tool_calls": len(tools), "edit_calls": edit_calls, "shell_edit_hints": shell_edits,
            "repeated_identical_cmds": rep, "tool_errors": errors,
            "successful_tool_calls": ok_calls, "rejected_tool_calls": rejected,
            "acknowledged_source_edit_calls": ack_edits, "tmp_scratch_writes": scratch_writes,
            "repeated_identical_tool_calls": rep_tools,
            "unparsed_tool_call_texts": unparsed_texts,
            # DESCRIPTIVE flags, not a diagnosis. Each names an observed pattern in the trace.
            # "rejected_calls" and "unparsed_text" together are a symptom of tool-use trouble; they
            # are NOT evidence of any particular root cause, encoding or otherwise.
            "tool_use_flags": ";".join(f for f, on in (
                ("rejected_calls", rejected > 0),
                ("unparsed_text", unparsed_texts > 0),
                ("identical_repeats", rep_tools > 0),
                ("no_acknowledged_source_edit", ack_edits == 0),
            ) if on) or "none",
            "first_source_edit_attempt_step": first_edit if first_edit is not None else UNAVAILABLE,
            "finish_reasons": ";".join(sorted(set(finish))) or UNAVAILABLE,
            "prompt_tokens": fm.get("total_prompt_tokens", UNAVAILABLE),
            "completion_tokens": fm.get("total_completion_tokens", UNAVAILABLE)}

def patch_shape(patch):
    """What the diff actually contains, per diff block. A shell edit and an edit_file edit are
    indistinguishable here, which is the point: absence of edit_file is not absence of a fix."""
    files, new_files = [], set()
    is_new = False
    for line in (patch or "").splitlines():
        if line.startswith("diff --git"):
            is_new = False
        elif line.startswith("new file mode"):
            is_new = True
        elif line.startswith("--- ") and line.strip().endswith("/dev/null"):
            is_new = True
        elif line.startswith("+++ b/"):
            path = line[6:].strip()
            if path and path != "/dev/null":
                files.append(path)
                if is_new:
                    new_files.add(path)
    modifies_existing = any(f not in new_files for f in files)
    adds_nested = any(f in new_files and "/" in f for f in files)
    return {"patch_files": files, "patch_new_files": sorted(new_files),
            "patch_modifies_existing": modifies_existing,
            "patch_touches_source": any(Path(f).name not in {"repro.py", "debug.py"} for f in files)}

def attribute(row, patch, tr_stats, shape):
    if row.get("resolved"):
        return "passed"
    err = (row.get("error") or "")
    # Distinguish "the agent never submitted" from "a patch was produced and was empty". Both leave
    # no grade, and neither is a graded loss.
    if "completed execution without calling submit_patch" in err.lower():
        return "no_submission"
    if not patch.strip():
        return "empty_patch"
    if not shape["patch_touches_source"]:
        return "suspected_scratch_only_patch"
    if "Failed to apply" in err:
        return "patch_apply_failed"
    if "timeout" in err.lower() or "exceeded session" in err.lower():
        return "budget_time"
    if "turns budget" in err.lower() or "tool call budget" in err.lower():
        return "budget_turns_or_tools"
    return "tests_failed"

# Every planned run appears exactly once, executed or not. An early stop must not shrink the
# report: a missing row is indistinguishable from a run that was never planned.
ROW_KEYS = ["candidate", "task", "attempted", "stop_reason", "resolved", "attribution",
            "harness_error", "test_exit_code", "grading_ran", "grading_observed",
            "outcome", "outcome_reason", "cleanup_ok", "owned_sandboxes",
            "patch_bytes", "patch_touches_source", "patch_files", "wall_s",
            "agent_phase_s", "grading_phase_s", "both_setup_imports_verified", "phase_status",
            "failure_class", "agent_loop_s", "termination_error", "duration_seconds_harness",
            "tool_calls", "edit_calls", "shell_edit_hints", "repeated_identical_cmds",
            "tool_errors", "first_source_edit_attempt_step", "finish_reasons",
            "prompt_tokens", "completion_tokens",
            "successful_tool_calls", "rejected_tool_calls", "acknowledged_source_edit_calls",
            "tmp_scratch_writes", "repeated_identical_tool_calls", "unparsed_tool_call_texts",
            "tool_use_flags",
            "run_error"]

_runs = RUNS if "RUNS" in dir() else []
RUN_BY_KEY = {}
for _r in _runs:
    _k = (_r["task"], _r["candidate"])
    assert _k not in RUN_BY_KEY, f"duplicate run record for {_k}"
    RUN_BY_KEY[_k] = _r
PLANNED = list(ORDER) if "ORDER" in dir() else []
assert len(PLANNED) == len(set(PLANNED)), "ORDER contains duplicate (task, candidate) pairs"
_orphans = set(RUN_BY_KEY) - set(PLANNED)
assert not _orphans, f"run records outside the planned ORDER: {sorted(_orphans)}"
_stop = STOP_REASON if "STOP_REASON" in dir() else None

rows = []
for _tid, _cand in PLANNED:
    r = RUN_BY_KEY.get((_tid, _cand))
    if r is None:
        row = {k: UNAVAILABLE for k in ROW_KEYS}
        row.update({"candidate": _cand, "task": _tid, "attempted": False,
                    "attribution": "not_attempted",
                    "stop_reason": _stop or "not reached before the run loop ended",
                    "failure_class": "not_attempted", "outcome": "not_attempted",
                    "run_error": "", "harness_error": UNAVAILABLE})
        assert set(row) == set(ROW_KEYS), "not_attempted row shape drifted from ROW_KEYS"
        rows.append(row)
        continue
    d = Path(r["dir"]); tid = r["task"]
    jf = d / "task_results.jsonl"
    raw = [json.loads(l) for l in jf.read_text(encoding="utf-8").splitlines() if l.strip()] if jf.exists() else []
    assert len(raw) <= 1, f'Duplicate results in {jf}'
    rec = raw[0] if raw else {}
    assert not rec or rec.get('instance_id') == tid, f'Wrong task result in {jf}' 
    safe = tid.replace("/", "__")
    pf = d / "patches" / (safe + ".patch")
    patch = pf.read_text(encoding="utf-8") if pf.exists() else ""
    tf = d / "test_outputs" / (safe + ".log")
    tests = tf.read_text(encoding="utf-8") if tf.exists() else ""
    ts = trace_stats(load_trace(d, tid))
    artifact_ok = not rec or len(patch) == rec.get('agent_patch_size')
    if rec and not artifact_ok:
        raise AssertionError(f'Patch artifact missing/mismatched for {tid}')
    shape = patch_shape(patch)
    agent_s = sum(r["agent_phase_s"]) if r["agent_phase_s"] else UNAVAILABLE
    grade_s = sum(r["grading_phase_s"]) if r["grading_phase_s"] else UNAVAILABLE
    setup_plus_agent = agent_s
    row = {
        "candidate": r["candidate"], "task": tid,
        "attempted": True, "stop_reason": "",
        "grading_observed": r.get("grading_observed", UNAVAILABLE),
        "outcome": r.get("outcome", UNAVAILABLE),
        "outcome_reason": r.get("outcome_reason", ""),
        "cleanup_ok": r.get("cleanup_ok", UNAVAILABLE),
        "owned_sandboxes": ";".join(r.get("owned_sandboxes") or []) or "",
        "resolved": rec.get("resolved", UNAVAILABLE),
        "attribution": attribute(rec, patch, ts, shape) if rec else "run_error",
        "harness_error": (rec.get("error") or "")[:120] or UNAVAILABLE,
        "test_exit_code": rec.get("test_exit_code", UNAVAILABLE),
        "grading_ran": bool(tests.strip()) and isinstance(rec.get("test_exit_code"), int) and rec["test_exit_code"] >= 0 if rec else UNAVAILABLE,
        "patch_bytes": len(patch.encode("utf-8")),
        "patch_touches_source": shape["patch_touches_source"],
        "patch_files": ";".join(shape["patch_files"]) or UNAVAILABLE,
        "wall_s": r["wall_s"], "agent_phase_s": setup_plus_agent, "grading_phase_s": grade_s,
        "both_setup_imports_verified": r.get('both_setup_imports_verified', UNAVAILABLE),
        "phase_status": json.dumps(r.get('phase_status', UNAVAILABLE)),
        "failure_class": ("provenance" if r.get('provenance_failed') else
                          "environment" if r.get('environment_error') else
                          "unobserved_grading" if r.get('unobserved_grading') else
                          "candidate" if r.get('candidate_error') else "none"),
        "agent_loop_s": sum(r.get('agent_loop_s', [])) if r.get('agent_loop_s') else UNAVAILABLE,
        "termination_error": rec.get('error') or UNAVAILABLE,
        "duration_seconds_harness": rec.get("duration_seconds", UNAVAILABLE),
        **ts, "run_error": (r.get("error") or "")[:120] or "",
    }
    missing, extra = set(ROW_KEYS) - set(row), set(row) - set(ROW_KEYS)
    assert not missing and not extra, f"row shape drift: missing={missing} extra={extra}"
    rows.append(row)

if rows:
    with (WORKING_DIR / "pilot_results.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    for r in rows:
        print(json.dumps(r))
    print("")
    print("%-10s %-16s %-9s %-22s %s" % ("cand", "task", "resolved", "attribution", "agent_s"))
    for r in rows:
        print("%-10s %-16s %-9s %-22s %s" % (r["candidate"], r["task"], r["resolved"],
                                             r["attribution"], r["agent_phase_s"]))
    print("")
    print("Four tasks from ONE repository (Textualize/rich) are a narrow diagnostic, not a general")
    print("ranking: any result here generalises to rich at best, and cannot say which candidate is")
    print("better overall. Both preconditions and evaluation run on the SUBPROCESS backend, which is")
    print("not a filesystem isolation boundary. agent_phase_s covers container-A setup plus the agent loop;")
    print("grading_phase_s is container B. The harness duration_seconds bundles both and must not be")
    print("multiplied to project the 12-hour scoring runtime.")
else:
    print("no runs executed (DISPATCH_CONFIRM was False)")

MANIFEST = {
    "candidates": {n: b["sha256"] for n, b in BUNDLES.items()},
    "candidate_dirs": {n: str(d) for n, d in CAND_DIRS.items()},
    "tasks": TASK_IDS, "control_evidence": CONTROL_EVIDENCE,
    "expected_task_hashes": EXPECTED_TASK_HASHES,
    "preconditions": PRECONDITIONS if "PRECONDITIONS" in dir() else UNAVAILABLE,
    "budgets": BUDGET, "seed_in_sampling": 42,
    "package_versions": VERSIONS, "model_config_sha256": MODEL_CONFIG_SHA,
    "model_path": str(MODEL_PATH), "quantization": QUANT,
    "gpus": {"count": N_GPU, "names": GPU_NAMES},
    "run_order": ORDER if "ORDER" in dir() else UNAVAILABLE,
    "repair_log": REPAIR_LOG,
    "session_cap_min": SESSION_CAP_MIN,
    "session_cap_kind": "admission limit; no hard kill of in-flight processes",
    "run_reserve_min": RUN_RESERVE_MIN,
    "server_stopped": SERVER_STOPPED,
    "setup_observation_note": "Fresh import probes with agent and grading flags; not direct in-pytest instrumentation",
    "task_content_hashes": {t.instance_id: hashlib.sha256(json.dumps({k:getattr(t,k,None) for k in
        ('repo','base_commit','problem_statement','hints_text','patch','test_patch')},sort_keys=True).encode()).hexdigest() for t in SELECTED},
    "notes": ["thinking_budget is NOT forwarded to the server; only enable_thinking is",
              "subprocess backend is not a filesystem isolation boundary",
              "repair is evaluation support only and is not part of any submission artifact"],
}
(WORKING_DIR / "pilot_manifest.json").write_text(json.dumps(MANIFEST, indent=2, default=str), encoding="utf-8")
print("saved pilot_manifest.json and pilot_results.csv")
